# Hướng dẫn truy cập và fix cứng môi trường bài toán Portfolios Management và các tinh chỉnh cấu hình

## Khởi tạo môi trường

In [ ]:
## install finrl library
!sudo apt install swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [ ]:
## We also need to install quantstats, because the environment uses it to plot graphs
!pip install quantstats

In [ ]:
## Hide matplotlib warnings
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

## Lớp khởi tạo môi trường chính của FinRL

---

Notebook này nhằm mục đích cung cấp một ví dụ về việc sử dụng PortfolioOptimizationEnv (hoặc POE) để huấn luyện một mô hình học tăng cường (reinforcement learning) nhằm giải quyết bài toán tối ưu hóa danh mục đầu tư.

Trong tài liệu này, chúng ta sẽ tái hiện một kiến trúc nổi tiếng có tên là EIIE (Ensemble of Identical Independent Evaluators - tập hợp các bộ đánh giá độc lập giống hệt nhau), được giới thiệu trong bài báo sau:

* Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. [https://doi.org/10.48550/arXiv.1706.10059](https://doi.org/10.48550/arXiv.1706.10059).

Bạn nên đọc bài báo này để hiểu rõ thuật toán được triển khai trong notebook.

### Lưu ý

Nếu bạn sử dụng môi trường này, hãy cân nhắc trích dẫn bài báo sau (bên cạnh các tài liệu tham khảo về FinRL):

* Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (tr. 132–143). SBC. [https://doi.org/10.5753/bwaif.2023.231144](https://doi.org/10.5753/bwaif.2023.231144).

```bibtex
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

In [ ]:
!pip install torch_geometric

In [ ]:
from typing import Optional, List, Dict, Union, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.preprocessing import MaxAbsScaler

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

## Điều chỉnh các đầu vào custom có thể diễn ra

### - Đầu vào chuẩn không áp dụng cho phân tích cảm xúc thị trường (Sentiment Analysis)


In [ ]:
# ====================================================================
# ÁP DỤNG TRỰC TIẾP MÔI TRƯỜNG PortfolioOptimizationEnv CHÍNH THỨC CỦA FinRL
# (Costa & Costa 2023, BWAIF - Kế thừa trực tiếp, KHÔNG rewrite lại môi trường)
# ====================================================================

from typing import Optional, List, Dict, Union, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv

class PortfolioOptimizationEnv_v0(PortfolioOptimizationEnv):
    """
    Kế thừa và áp dụng trực tiếp PortfolioOptimizationEnv chính thức của FinRL (Costa & Costa, 2023)
    đã được công bố và kiểm chứng trong file FinRL_PortfolioOptimizationEnv_Demo.ipynb.
    
    Môi trường mô phỏng đầy đủ:
    - Vector biến động giá tương đối y_t = [1.0, p_t / p_{t-1}]
    - Mô hình phí bảo toàn vốn TRF (Transaction Remainder Factor, Jiang et al. 2017)
    - Trôi dạt trọng số thị trường (Price Drift)
    - Phần thưởng log-return: r_t = ln(mu_t * y_t^T * w_t)
    - Không gian quan sát Tensor 3D (F x N x M) và Action Simplex (N + 1)
    """
    def __init__(
        self,
        df: Optional[pd.DataFrame] = None,
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,
        features: Optional[List[str]] = None,
        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,
        # Các tham số hỗ trợ linh hoạt cho cả DataFrame FinRL và ma trận giá
        df_features: Optional[np.ndarray] = None,
        df_prices: Optional[np.ndarray] = None,
        tic_list: Optional[List[str]] = None,
        fee_rate: Optional[float] = None,
        fee_model: str = "trf",
        reward_scaling: float = 1.0,
        **kwargs
    ):
        if features is None:
            features = ["close", "high", "low"]
        if fee_rate is not None:
            comission_fee_pct = fee_rate

        # Chuyển đổi dữ liệu sang định dạng DataFrame chuẩn FinRL nếu truyền ma trận giá
        if df is None and df_prices is not None:
            T_len, N_len = df_prices.shape
            if tic_list is None:
                tic_list = [f"TIC_{i}" for i in range(N_len)]
            date_range = pd.date_range("2020-01-01", periods=T_len, freq="B").strftime("%Y-%m-%d")
            records = []
            for t_idx, d_str in enumerate(date_range):
                for n_idx, tic in enumerate(tic_list):
                    close_p = float(df_prices[t_idx, n_idx])
                    records.append({
                        "date": d_str,
                        "tic": tic,
                        "close": close_p,
                        "high": close_p * 1.002,
                        "low": close_p * 0.998
                    })
            df = pd.DataFrame(records)

        # Gọi trực tiếp khởi tạo của PortfolioOptimizationEnv từ FinRL (KHÔNG rewrite)
        super().__init__(
            df=df,
            initial_amount=initial_amount,
            return_last_action=return_last_action,
            normalize_df=normalize_df,
            reward_scaling=reward_scaling,
            comission_fee_model=fee_model,
            comission_fee_pct=comission_fee_pct,
            features=features,
            time_window=time_window,
            new_gym_api=new_gym_api,
            **kwargs
        )

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Khởi tạo lại môi trường tương thích Gymnasium API mới."""
        if seed is not None:
            self._seed(seed)
        res = super().reset()
        if not self._new_gym_api and not isinstance(res, tuple):
            return res, {}
        return res

    def step(self, actions: np.ndarray):
        """Thực thi một bước mô phỏng với PortfolioOptimizationEnv chính thức."""
        res = super().step(actions)
        if len(res) == 5:
            obs, reward, terminal, truncated, info = res
        else:
            obs, reward, terminal, info = res
            truncated = False
        # Bổ sung các trường tiện ích cho tương thích đánh giá trong notebook
        info["fee_factor"] = info.get("trf_mu", 1.0)
        info["portfolio_value"] = self._portfolio_value
        return obs, reward, terminal, truncated, info

    @property
    def action_dim(self) -> int:
        return self.action_space.shape[0]

    @property
    def num_stocks(self) -> int:
        return self.portfolio_size

    @property
    def time_window(self) -> int:
        return self._time_window

    @property
    def num_features(self) -> int:
        return len(self._features)

    @property
    def T(self) -> int:
        return self.episode_length

    @property
    def asset_memory(self) -> list:
        return self._asset_memory.get("final", [self._initial_amount])

    def compute_metrics(self) -> Dict[str, float]:
        """Tính toán các chỉ số tài chính đánh giá hiệu suất danh mục."""
        values = np.array(self.asset_memory)
        if len(values) < 2:
            return {"Final Value ($)": self._portfolio_value}
        returns = np.diff(values) / values[:-1]
        total_return = (values[-1] - values[0]) / values[0]
        ann_return = ((1.0 + total_return) ** (252.0 / len(values))) - 1.0 if len(values) > 0 else 0.0
        ann_volatility = np.std(returns) * np.sqrt(252.0) if len(returns) > 1 else 0.0
        sharpe = (ann_return - 0.02) / (ann_volatility + 1e-8) if ann_volatility > 0 else 0.0
        cummax = np.maximum.accumulate(values)
        drawdowns = (cummax - values) / cummax
        max_drawdown = np.max(drawdowns) if len(drawdowns) > 0 else 0.0
        return {
            'Initial Value ($)': self._initial_amount,
            'Final Value ($)': values[-1],
            'Cumulative Return (%)': total_return * 100.0,
            'Annualized Return (%)': ann_return * 100.0,
            'Annualized Volatility (%)': ann_volatility * 100.0,
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': max_drawdown * 100.0
        }

    def plot_results(self, title: str = "Portfolio Value Over Time"):
        """Hiển thị đồ thị tăng trưởng giá trị danh mục."""
        values = self.asset_memory
        plt.figure(figsize=(10, 5))
        plt.plot(values, label="Portfolio Value", color="#1f77b4", lw=2.0)
        plt.title(title, fontsize=12, fontweight="bold")
        plt.xlabel("Trading Steps", fontsize=10)
        plt.ylabel("Portfolio Value ($)", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10)
        plt.show()

print('✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!')

### - Đầu vào cấu hình 1 HISTORICAL PRICE ENVIRONMENT

In [ ]:
# ====================================================================
# V1 - HISTORICAL PRICE ENVIRONMENT
#
# Paper:
#   Multimodal Deep Reinforcement Learning for Portfolio Optimization
#
# Configuration:
#   Config 1 - Historical Price Data
#
# Parent:
#   PortfolioOptimizationEnv_v0
#
# Customization:
#   - Override __init__()
#   - Restrict observation features to historical prices
#
# Preserved:
#   - FinRL reset()
#   - FinRL step()
#   - Portfolio accounting
#   - Price drift
#   - TRF transaction cost
#   - Original reward
# ====================================================================

import pandas as pd
from typing import Optional, List


class PortfolioOptimizationEnv_v1(PortfolioOptimizationEnv_v0):

    """
    V1: Historical Price Configuration.

    INPUT:
        df: pandas.DataFrame

        Required columns:
            date
            tic
            close
            high
            low

        Data requirements:
            - Each row represents one asset at one timestamp.
            - All assets must have consistent chronological ordering.
            - Prices must be valid and finite.
            - No sentiment features are included.
            - Data must not contain future information.

    OBSERVATION:
        Historical price tensor generated by FinRL.

        Expected conceptual dimensions:
            (F, N, H)

        F = 3 price features
        N = number of assets
        H = time_window

        Actual axis order must be verified against FinRL runtime.

    ACTION:
        Same action space as V0.

    OUTPUT:
        reset():
            observation, info

        step():
            observation,
            reward,
            terminated,
            truncated,
            info

        Portfolio dynamics and reward remain inherited from V0.
    """

    # ---------------------------------------------------------------
    # V1 INPUT CONFIGURATION
    # ---------------------------------------------------------------

    V1_FEATURES = [
        "close",
        "high",
        "low",
    ]

    REQUIRED_COLUMNS = [
        "date",
        "tic",
        "close",
        "high",
        "low",
    ]

    # ---------------------------------------------------------------
    # OVERRIDE __init__
    # ---------------------------------------------------------------

    def __init__(
        self,
        df: pd.DataFrame,

        # FinRL original settings
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,

        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,

        **kwargs
    ):

        # ===========================================================
        # INPUT VALIDATION
        # ===========================================================

        if df is None:
            raise ValueError("V1 requires a DataFrame.")

        missing = [
            col for col in self.REQUIRED_COLUMNS
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"V1 missing columns: {missing}"
            )

        # ===========================================================
        # V1 CUSTOM FEATURES
        # ===========================================================

        features = self.V1_FEATURES.copy()

        # ===========================================================
        # INHERIT FINRL ENVIRONMENT
        # ===========================================================

        super().__init__(
            df=df,
            initial_amount=initial_amount,
            comission_fee_pct=comission_fee_pct,
            time_window=time_window,
            features=features,
            normalize_df=normalize_df,
            return_last_action=return_last_action,
            new_gym_api=new_gym_api,
            **kwargs
        )

        # -----------------------------------------------------------
        # V1 METADATA
        # -----------------------------------------------------------

        self.version = "v1"
        self.configuration = "historical_price"
        self.custom_features = features

### - Đầu vào cấu hình 2 HISTORICAL PRICE + SEC SENTIMENT ENVIRONMENT

In [ ]:
# ====================================================================
# V2 - PRICE + SEC SENTIMENT ENVIRONMENT
#
# Paper:
#   Multimodal Deep Reinforcement Learning for Portfolio Optimization
#
# Configuration:
#   Config 2 - Historical Price + SEC Sentiment
#
# Parent:
#   PortfolioOptimizationEnv_v0
#
# Customization:
#   - Override __init__()
#   - Add SEC sentiment feature
#
# Preserved:
#   - FinRL portfolio dynamics
#   - TRF transaction cost
#   - Price drift
#   - FinRL reward
#   - FinRL action processing
# ====================================================================


import pandas as pd
from typing import Optional, List


class PortfolioOptimizationEnv_v2(PortfolioOptimizationEnv_v0):

    """
    V2: Price + SEC Sentiment.

    INPUT:
        pandas.DataFrame

        Required columns:
            date
            tic
            close
            high
            low
            sec_sentiment

    SEC INPUT RULES:
        - Sentiment must be computed before environment initialization.
        - SEC scores must be aligned by ticker and date.
        - Sentiment must be available at decision time.
        - No future filing information is allowed.
        - Missing-date values should follow the paper's
          exponential decay procedure.

    OBSERVATION:
        Historical price features + SEC sentiment.

        Conceptual dimensions:
            (F, N, H)

        F = 4 features
        N = number of assets
        H = time_window

    ACTION:
        Same as V0.

    REWARD:
        Same as V0.

    OUTPUT:
        Same Gymnasium interface as V0.
    """

    # ---------------------------------------------------------------
    # V2 CUSTOM FEATURES
    # ---------------------------------------------------------------

    V2_FEATURES = [
        "close",
        "high",
        "low",
        "sec_sentiment",
    ]

    REQUIRED_COLUMNS = [
        "date",
        "tic",
        "close",
        "high",
        "low",
        "sec_sentiment",
    ]

    # ---------------------------------------------------------------
    # OVERRIDE __init__
    # ---------------------------------------------------------------

    def __init__(
        self,
        df: pd.DataFrame,

        # FinRL original settings
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,

        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,

        **kwargs
    ):

        # ===========================================================
        # INPUT VALIDATION
        # ===========================================================

        if df is None:
            raise ValueError("V2 requires a DataFrame.")

        missing = [
            col for col in self.REQUIRED_COLUMNS
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"V2 missing columns: {missing}"
            )

        # ===========================================================
        # SEC SENTIMENT VALIDATION
        # ===========================================================

        if df["sec_sentiment"].isna().any():
            raise ValueError(
                "V2 requires SEC sentiment values to be "
                "preprocessed and aligned."
            )

        # ===========================================================
        # V2 CUSTOM FEATURES
        # ===========================================================

        features = self.V2_FEATURES.copy()

        # ===========================================================
        # INHERIT FINRL ENVIRONMENT
        # ===========================================================

        super().__init__(
            df=df,
            initial_amount=initial_amount,
            comission_fee_pct=comission_fee_pct,
            time_window=time_window,
            features=features,
            normalize_df=normalize_df,
            return_last_action=return_last_action,
            new_gym_api=new_gym_api,
            **kwargs
        )

        # -----------------------------------------------------------
        # V2 METADATA
        # -----------------------------------------------------------

        self.version = "v2"
        self.configuration = "price_sec"
        self.custom_features = features

### - Đầu vào cấu hình 3 HISTORICAL PRICE + SEC + NEWS SENTIMENT ENVIRONMENT

In [ ]:
# ====================================================================
# V3 - PRICE + SEC + NEWS SENTIMENT ENVIRONMENT
#
# Paper:
#   Multimodal Deep Reinforcement Learning for Portfolio Optimization
#
# Configuration:
#   Config 3 - Historical Price + SEC + News Sentiment
#
# Parent:
#   PortfolioOptimizationEnv_v0
#
# Customization:
#   - Override __init__()
#   - Add SEC sentiment
#   - Add News sentiment
#
# Preserved:
#   - FinRL portfolio accounting
#   - TRF transaction cost
#   - Price drift
#   - Original reward
#   - Original action processing
# ====================================================================


import pandas as pd
from typing import Optional, List


class PortfolioOptimizationEnv_v3(PortfolioOptimizationEnv_v0):

    """
    V3: Price + SEC + News Sentiment.

    INPUT:
        pandas.DataFrame

        Required columns:
            date
            tic
            close
            high
            low
            sec_sentiment
            news_sentiment

    SEC INPUT:
        - SEC sentiment must be computed in preprocessing.
        - SEC score must be aligned by ticker/date.
        - Exponential decay must be applied according to
          the paper's preprocessing procedure.

    NEWS INPUT:
        - News sentiment must be generated before environment
          initialization.
        - FinBERT probabilities are processed into sentiment score.
        - News sentiment must be aligned by ticker/date.
        - Exponential decay handles missing news dates.
        - Future news must not enter current observations.

    OBSERVATION:
        Historical prices + SEC + News.

        Conceptual dimensions:
            (F, N, H)

        F = 5 features
        N = number of assets
        H = time_window

    ACTION:
        Same as V0.

    REWARD:
        Same as V0.

    OUTPUT:
        Same Gymnasium interface as V0.
    """

    # ---------------------------------------------------------------
    # V3 CUSTOM FEATURES
    # ---------------------------------------------------------------

    V3_FEATURES = [
        "close",
        "high",
        "low",
        "sec_sentiment",
        "news_sentiment",
    ]

    REQUIRED_COLUMNS = [
        "date",
        "tic",
        "close",
        "high",
        "low",
        "sec_sentiment",
        "news_sentiment",
    ]

    # ---------------------------------------------------------------
    # OVERRIDE __init__
    # ---------------------------------------------------------------

    def __init__(
        self,
        df: pd.DataFrame,

        # FinRL original settings
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.01,
        time_window: int = 30,

        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,

        **kwargs
    ):

        # ===========================================================
        # INPUT VALIDATION
        # ===========================================================

        if df is None:
            raise ValueError("V3 requires a DataFrame.")

        missing = [
            col for col in self.REQUIRED_COLUMNS
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"V3 missing columns: {missing}"
            )

        # ===========================================================
        # SENTIMENT VALIDATION
        #
        # Data preprocessing must define how missing sentiment
        # values are handled.
        #
        # Do not silently use future sentiment.
        # ===========================================================

        sentiment_columns = [
            "sec_sentiment",
            "news_sentiment",
        ]

        for col in sentiment_columns:

            if df[col].isna().any():

                raise ValueError(
                    f"V3 requires preprocessing of missing "
                    f"sentiment values in '{col}'."
                )

        # ===========================================================
        # V3 CUSTOM FEATURES
        # ===========================================================

        features = self.V3_FEATURES.copy()

        # ===========================================================
        # INHERIT FINRL ENVIRONMENT
        # ===========================================================

        super().__init__(
            df=df,
            initial_amount=initial_amount,
            comission_fee_pct=comission_fee_pct,
            time_window=time_window,
            features=features,
            normalize_df=normalize_df,
            return_last_action=return_last_action,
            new_gym_api=new_gym_api,
            **kwargs
        )

        # -----------------------------------------------------------
        # V3 METADATA
        # -----------------------------------------------------------

        self.version = "v3"
        self.configuration = "price_sec_news"
        self.custom_features = features

### - Môi trường đặc trưng cho FinRL kết hợp 2 thành phần phần thưởng mới

In [ ]:
import numpy as np
import pandas as pd

from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import (
    PortfolioOptimizationEnv
)


class PortfolioZeroRewardPOE(PortfolioOptimizationEnv):
    """
    FinRL PortfolioOptimizationEnv + PortfolioZero-style sentiment rewards.

    Kế thừa:
        - State, reset, action normalization
        - Transaction fee
        - Portfolio value accounting
        - Price variation / portfolio weights

    Override:
        - Reward R1, R2, R3
    """

    def __init__(
        self,
        *args,
        sentiment_df=None,
        reward_mode="R1",
        sentiment_weight=1.0,
        beta_weight=1.0,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        if reward_mode not in {"R1", "R2", "R3"}:
            raise ValueError("reward_mode phải là R1, R2 hoặc R3")

        self.reward_mode = reward_mode
        self.sentiment_weight = sentiment_weight
        self.beta_weight = beta_weight

        # Data format: date, tic, score
        if sentiment_df is None:
            sentiment_df = pd.DataFrame(
                columns=["date", "tic", "score"]
            )

        sentiment_df = sentiment_df.copy()
        sentiment_df["date"] = pd.to_datetime(
            sentiment_df["date"]
        )

        self.sentiment_df = sentiment_df

        # Lookup: (date, ticker) -> sentiment score
        self.sentiment_lookup = {
            (row.date, row.tic): float(row.score)
            for row in sentiment_df.itertuples(index=False)
        }

        # Logging reward components
        self.reward_components = []

    def _get_sentiment_vector(self, date):
        """
        Trả về sentiment theo đúng thứ tự self._tics.

        POE action convention:
            index 0 = cash
            index 1..N = risky assets

        Cash sentiment được gán bằng 0.
        """
        date = pd.to_datetime(date)

        sentiment = np.zeros(len(self._tics) + 1, dtype=np.float32)

        for i, tic in enumerate(self._tics, start=1):
            sentiment[i] = self.sentiment_lookup.get(
                (date, tic), 0.0
            )

        return sentiment

    @staticmethod
    def _rank_weights_change(delta_weights):
        """
        Rank theo mức thay đổi weights.

        Rank nhỏ: mức thay đổi thấp hơn.
        Rank lớn: mức thay đổi cao hơn.

        Ties được xử lý bằng average rank.
        """
        return (
            pd.Series(delta_weights)
            .rank(method="average")
            .to_numpy(dtype=np.float32)
        )

    def _calculate_sentiment_rewards(
        self,
        weights,
        previous_weights,
        sentiment
    ):
        """
        weights:
            Target weights agent vừa yêu cầu.

        previous_weights:
            Weights trước khi thực hiện action.

        sentiment:
            Sentiment vector gồm cash + stocks.
        """

        # Exclude cash index 0
        stock_weights = weights[1:]
        stock_previous_weights = previous_weights[1:]
        stock_sentiment = sentiment[1:]

        # R_alpha: sentiment-weighted allocation
        R_alpha = np.sum(
            stock_weights * stock_sentiment
        )

        # Delta portfolio weights
        delta_weights = (
            stock_weights - stock_previous_weights
        )

        # Rank delta weights
        ranks = self._rank_weights_change(delta_weights)

        # R_beta: pairwise sentiment differences * rank differences
        sentiment_diff = (
            stock_sentiment[:, None]
            - stock_sentiment[None, :]
        )

        rank_diff = (
            ranks[:, None]
            - ranks[None, :]
        )

        R_beta = np.sum(
            sentiment_diff * rank_diff
        )

        return float(R_alpha), float(R_beta)

    def step(self, actions):
        """
        POE step + custom PortfolioZero reward.

        Base POE vẫn xử lý:
            action normalization
            transaction fees
            portfolio value
            price variation
            next state
            terminal condition
        """

        # Base environment step
        result = super().step(actions)

        # Gym API compatibility
        if self._new_gym_api:
            state, base_reward, terminal, truncated, info = result
        else:
            state, base_reward, terminal, info = result

        # Nếu POE đã terminal thì không có action trading mới
        if terminal:
            return result

        # Base reward trước reward_scaling
        # PortfolioZero R = log(V_t / V_{t-1})
        base_log_return = np.log(
            self._asset_memory["final"][-1]
            / self._asset_memory["final"][-2]
        )

        # Current target action weights
        current_weights = np.asarray(
            self._actions_memory[-1],
            dtype=np.float32
        )

        # Portfolio weights before action
        previous_weights = np.asarray(
            self._final_weights[-2],
            dtype=np.float32
        )

        # Sentiment date corresponding to this transition
        date = pd.to_datetime(info["end_time"])

        sentiment = self._get_sentiment_vector(date)

        R_alpha, R_beta = self._calculate_sentiment_rewards(
            weights=current_weights,
            previous_weights=previous_weights,
            sentiment=sentiment
        )

        # Reward composition
        if self.reward_mode == "R1":
            reward_unscaled = base_log_return

        elif self.reward_mode == "R2":
            reward_unscaled = (
                base_log_return
                + self.sentiment_weight * R_alpha
            )

        elif self.reward_mode == "R3":
            reward_unscaled = (
                base_log_return
                + self.sentiment_weight * R_alpha
                + self.beta_weight * R_beta
            )

        # Apply POE reward scaling exactly once
        reward = reward_unscaled * self._reward_scaling

        # Override reward from base POE
        self._reward = reward

        # Update reward memory so POE's episode metrics use custom reward
        self._portfolio_reward_memory[-1] = reward_unscaled

        # Logging
        reward_log = {
            "date": date,
            "base_log_return": float(base_log_return),
            "R_alpha": float(R_alpha),
            "R_beta": float(R_beta),
            "reward_unscaled": float(reward_unscaled),
            "reward_scaled": float(reward),
            "reward_mode": self.reward_mode
        }

        self.reward_components.append(reward_log)

        # Add components to info
        info = dict(info)
        info.update(reward_log)

        if self._new_gym_api:
            return state, reward, terminal, truncated, info

        return state, reward, terminal, info

## Truyền mô hình và scoring sentiment

In [ ]:
import torch
import math
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Khởi tạo mô hình và tokenizer FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
def calculate_custom_sentiment_score(text):
    # 2. Tiền xử lý văn bản
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    
    # 3. Đưa qua mô hình để lấy raw logits
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
    # 4. Áp dụng hàm Softmax để tính xác suất (probabilities)
    probabilities = torch.nn.functional.softmax(logits, dim=-1).squeeze().tolist()
    
    # Thứ tự nhãn của ProsusAI/finbert là: [positive, negative, neutral]
    # Lưu ý: Bạn cần kiểm tra lại model.config.id2label để chắc chắn thứ tự nhãn
    positive_prob = probabilities[0]
    negative_prob = probabilities[1]
    neutral_prob = probabilities[2]
    
    # Xử lý trường hợp negative_prob hoặc neutral_prob quá nhỏ (tránh chia cho 0)
    epsilon = 1e-9
    negative_prob = max(negative_prob, epsilon)
    neutral_prob = max(neutral_prob, epsilon)
    
    # 5. Áp dụng công thức (1) từ bài báo
    ratio = (positive_prob / negative_prob) / neutral_prob
    value_embedding = math.tanh(ratio)
    
    return {
        "text": text,
        "positive_prob": positive_prob,
        "negative_prob": negative_prob,
        "neutral_prob": neutral_prob,
        "scoring_value": value_embedding
    }

In [ ]:
# --- Chạy thử trên một tập mẫu (Sample) ---
sample_headlines = [
    "Apple iPhone sales drop at recent pace!",
    "Powerful 11th Generation Intel Core CPUs Are Coming.",
    "The company's quarterly revenue met analysts' expectations."
]

for headline in sample_headlines:
    result = calculate_custom_sentiment_score(headline)
    print(f"Headline: '{result['text']}'")
    print(f"  - Positive: {result['positive_prob']:.4f}")
    print(f"  - Negative: {result['negative_prob']:.4f}")
    print(f"  - Neutral:  {result['neutral_prob']:.4f}")
    print(f"  -> Scoring Value (value_embedding): {result['scoring_value']:.4f}\\n")

In [ ]:
!pip install "yfinance>=0.2.51,<0.3.0" "curl_cffi<=0.9.4"

In [ ]:
# ====================================================================
# CELL 14: TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU GIÁ 10 CỔ PHIẾU (ĐÃ SỬA LỖI CHROME150)
# ====================================================================

import time
import requests
import pandas as pd
import numpy as np

# 1. Danh sách 10 mã cổ phiếu mục tiêu
TARGET_TICKERS = ['GILD', 'NVDA', 'TSLA', 'MRK', 'KO', 'AAPL', 'GS', 'WMT', 'FDX', 'AMT']
FEATURES = ['close', 'high', 'low', 'open']
TIME_WINDOW = 30  # M = 30 ngày

TRAIN_START = '2019-01-01'
TRAIN_END   = '2022-12-31'
TEST_START  = '2023-01-01'
TEST_END    = '2023-12-31'

# Lấy dôi ra 60 ngày trước TRAIN_START để đủ cửa sổ trượt (lookback window) cho ngày đầu tiên
fetch_start_dt = pd.to_datetime(TRAIN_START) - pd.Timedelta(days=TIME_WINDOW * 2)
period1 = int(fetch_start_dt.timestamp())
period2 = int(pd.to_datetime(TEST_END).timestamp())

print(f">>> Đang tải dữ liệu trực tiếp từ Yahoo Finance API cho {len(TARGET_TICKERS)} mã...")

# Thiết lập session với User-Agent chuẩn để tránh lỗi curl_cffi/impersonation
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
})

ticker_dfs = {}

for tic in TARGET_TICKERS:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{tic}?period1={period1}&period2={period2}&interval=1d&events=history"
    success = False
    
    for attempt in range(3):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                result = data['chart']['result'][0]
                timestamps = result['timestamp']
                quote = result['indicators']['quote'][0]
                
                df_tic = pd.DataFrame({
                    'date': pd.to_datetime(timestamps, unit='s').strftime('%Y-%m-%d'),
                    'open': quote['open'],
                    'high': quote['high'],
                    'low': quote['low'],
                    'close': quote['close']
                })
                
                # Làm sạch giá trị thiếu (Forward-fill rồi Backward-fill)
                df_tic = df_tic.ffill().bfill()
                df_tic['tic'] = tic
                
                # Đưa date về định dạng chuỗi chuẩn
                df_tic = df_tic.drop_duplicates(subset=['date']).set_index('date')
                ticker_dfs[tic] = df_tic
                print(f"  ✓ Tải thành công {tic}: {len(df_tic)} phiên giao dịch.")
                success = True
                break
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
            
    if not success:
        raise RuntimeError(f"Không thể tải dữ liệu cho mã {tic}. Vui lòng kiểm tra lại kết nối mạng!")

# 2. Tìm các ngày giao dịch chung cho tất cả 10 mã
common_dates = None
for tic in TARGET_TICKERS:
    if common_dates is None:
        common_dates = ticker_dfs[tic].index
    else:
        common_dates = common_dates.intersection(ticker_dfs[tic].index)

common_dates = common_dates.sort_values()

# 3. Gom dữ liệu về bảng DataFrame chuẩn của FinRL: [date, tic, close, high, low, open]
final_records = []
for d in common_dates:
    for tic in TARGET_TICKERS:
        row = ticker_dfs[tic].loc[d]
        final_records.append({
            'date': d,
            'tic': tic,
            'open': float(row['open']),
            'high': float(row['high']),
            'low': float(row['low']),
            'close': float(row['close'])
        })

df_prices = pd.DataFrame(final_records).sort_values(['date', 'tic']).reset_index(drop=True)

# 4. Tách tập Train và Test độc lập
df_train_prices = df_prices[(df_prices['date'] >= TRAIN_START) & (df_prices['date'] <= TRAIN_END)].reset_index(drop=True)
df_test_prices  = df_prices[(df_prices['date'] >= TEST_START) & (df_prices['date'] <= TEST_END)].reset_index(drop=True)

print("\n" + "=" * 70)
print("✅ HOÀN TẤT TIỀN XỬ LÝ DỮ LIỆU GIÁ:")
print(f" - Tổng số mã: {len(TARGET_TICKERS)} mã ({', '.join(TARGET_TICKERS)})")
print(f" - Tập Train: {len(df_train_prices['date'].unique())} ngày ({TRAIN_START} -> {TRAIN_END}) | {len(df_train_prices)} dòng")
print(f" - Tập Test:  {len(df_test_prices['date'].unique())} ngày ({TEST_START} -> {TEST_END}) | {len(df_test_prices)} dòng")
print("=" * 70)

df_train_prices.head(10)

In [ ]:
# ====================================================================
# CELL 15: TÍCH HỢP TIN TỨC THỰC TẾ & CHẤM ĐIỂM FINBERT (ĐÃ SỬA LỖI DUPLICATE COLS)
# ====================================================================

import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

news_file_path = '/kaggle/input/datasets/nguyennghia108/sentiment/top10_official_news_2013_2023.csv'
if not os.path.exists(news_file_path):
    raise FileNotFoundError(f"Không tìm thấy file: {news_file_path}")

print(f">>> Đang đọc file: {news_file_path}")
df_news_raw = pd.read_csv(news_file_path)
print(f" - Các cột hiện có trong file: {list(df_news_raw.columns)}")

# 1. Nhận diện chính xác từng cột duy nhất (tránh duplicate column name)
cols = list(df_news_raw.columns)

def find_best_col(keywords, available_cols):
    for kw in keywords:
        for c in available_cols:
            if kw.lower() == c.lower():
                return c
    for kw in keywords:
        for c in available_cols:
            if kw.lower() in c.lower():
                return c
    return None

date_col = find_best_col(['date', 'publish_date', 'timestamp', 'time', 'datetime'], cols)
tic_col  = find_best_col(['ticker', 'tic', 'symbol', 'stock', 'stock_symbol'], cols)
text_col = find_best_col(['headline', 'title', 'text', 'content', 'summary', 'news'], cols)

print(f" - Cột được chọn: Date='{date_col}', Ticker='{tic_col}', Content='{text_col}'")

if not date_col or not tic_col or not text_col:
    raise ValueError(f"Không nhận diện được đủ 3 cột. Hãy kiểm tra lại danh sách cột: {cols}")

# Trích xuất đúng 3 cột dạng Series độc lập để loại bỏ hoàn toàn duplicate index/columns
df_news = pd.DataFrame({
    'date_raw': df_news_raw[date_col].astype(str),
    'tic': df_news_raw[tic_col].astype(str).str.strip().str.upper(),
    'text': df_news_raw[text_col].astype(str)
})

# 2. Chuẩn hóa ngày (xử lý cả định dạng có timezone lẫn không timezone)
df_news['date'] = pd.to_datetime(df_news['date_raw'], errors='coerce', utc=True).dt.tz_localize(None).dt.strftime('%Y-%m-%d')
df_news = df_news.dropna(subset=['date', 'tic', 'text'])

# Lọc đúng 10 mã cổ phiếu mục tiêu
target_upper = [t.upper() for t in TARGET_TICKERS]
mask = df_news['tic'].isin(target_upper)
df_news = df_news[mask].reset_index(drop=True)

# Lọc theo khoảng ngày giao dịch của df_prices
min_date = df_prices['date'].min()
max_date = df_prices['date'].max()
df_news = df_news[(df_news['date'] >= min_date) & (df_news['date'] <= max_date)].reset_index(drop=True)

print(f" - Số lượng tin tức hợp lệ sau khi lọc: {len(df_news)} tin tức.")
if len(df_news) == 0:
    print("⚠️ Cảnh báo: Không có tin tức nào khớp với khoảng ngày giao dịch. Sẽ kích hoạt chế độ suy giảm mặc định.")

# 3. Chấm điểm bằng FinBERT theo batch
model.to(device)
model.eval()

def score_headlines_batch(texts, batch_size=64):
    scores = []
    epsilon = 1e-9
    for i in tqdm(range(0, len(texts), batch_size), desc="Scoring FinBERT"):
        batch_texts = [str(t)[:512] for t in texts[i:i + batch_size]]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
            
        for prob in probs:
            pos, neg, neu = max(prob[0], epsilon), max(prob[1], epsilon), max(prob[2], epsilon)
            ratio = (pos / neg) / neu
            score = np.tanh(ratio)
            # Nếu Negative > Positive thì gán dấu âm
            if neg > pos:
                score = -abs(score)
            else:
                score = abs(score)
            scores.append(float(score))
    return scores

if len(df_news) > 0:
    df_news['sentiment_score'] = score_headlines_batch(df_news['text'].tolist(), batch_size=64)
    # Gom nhóm tính trung bình theo ngày và theo mã
    daily_sentiment = df_news.groupby(['date', 'tic'])['sentiment_score'].mean().reset_index()
    sentiment_lookup_raw = {(row['date'], row['tic']): row['sentiment_score'] for _, row in daily_sentiment.iterrows()}
else:
    sentiment_lookup_raw = {}

# 4. Cơ chế Suy giảm số mũ (Exponential Decay) cho các ngày không có tin tức
all_trading_dates = sorted(df_prices['date'].unique())
decay_rate = 0.90  # Paper PortfolioZero (Section 4.1.2)

final_sentiment_records = []
current_memory = {tic: 0.0 for tic in TARGET_TICKERS}

for d in all_trading_dates:
    for tic in TARGET_TICKERS:
        key = (d, tic)
        if key in sentiment_lookup_raw:
            score = sentiment_lookup_raw[key]
            current_memory[tic] = score
        else:
            current_memory[tic] = current_memory[tic] * decay_rate
            
        final_sentiment_records.append({
            'date': d,
            'tic': tic,
            'score': float(current_memory[tic])
        })

df_sentiment = pd.DataFrame(final_sentiment_records)

print("\n" + "=" * 70)
print("✅ CELL 15 HOÀN TẤT THÀNH CÔNG:")
print(f" - Bảng sentiment chuẩn: {len(df_sentiment)} dòng ({len(all_trading_dates)} ngày x {len(TARGET_TICKERS)} mã)")
print(f" - Khoảng thời gian: {df_sentiment['date'].min()} -> {df_sentiment['date'].max()}")
print(f" - Thống kê điểm số: Min={df_sentiment['score'].min():.4f}, Max={df_sentiment['score'].max():.4f}, Mean={df_sentiment['score'].mean():.4f}")
print("=" * 70)

df_sentiment.head(10)

In [ ]:
# ====================================================================
# BẢN VÁ (HOTFIX) CHO LỚP PortfolioZeroRewardPOE: THÊM THUỘC TÍNH _tics
# ====================================================================

import numpy as np
import pandas as pd
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv

class PortfolioZeroRewardPOE_Fixed(PortfolioOptimizationEnv):
    """
    Kế thừa trực tiếp FinRL PortfolioOptimizationEnv + khắc phục lỗi thuộc tính _tics.
    Hỗ trợ đầy đủ các hàm phần thưởng R1, R2, R3 theo đúng công thức bài báo PortfolioZero.
    """
    def __init__(
        self,
        *args,
        sentiment_df=None,
        reward_mode="R1",
        sentiment_weight=1.0,
        beta_weight=1.0,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        if reward_mode not in {"R1", "R2", "R3"}:
            raise ValueError("reward_mode phải là 'R1', 'R2' hoặc 'R3'")

        self.reward_mode = reward_mode
        self.sentiment_weight = sentiment_weight
        self.beta_weight = beta_weight

        # TỰ ĐỘNG XÁC ĐỊNH DANH SÁCH MÃ CỔ PHIẾU _tics AN TOÀN CHO FinRL
        if hasattr(self, '_tic_list'):
            self._tics = list(self._tic_list)
        elif hasattr(self, 'tic_list'):
            self._tics = list(self.tic_list)
        elif hasattr(self, '_df') and 'tic' in self._df.columns:
            self._tics = list(self._df['tic'].unique())
        else:
            # Fallback lấy từ TARGET_TICKERS nếu có
            self._tics = list(TARGET_TICKERS)

        # Chuẩn bị bảng tra cứu cảm xúc
        if sentiment_df is None:
            sentiment_df = pd.DataFrame(columns=["date", "tic", "score"])

        sentiment_df = sentiment_df.copy()
        sentiment_df["date"] = pd.to_datetime(sentiment_df["date"]).dt.strftime('%Y-%m-%d')
        self.sentiment_df = sentiment_df

        # Lookup: (date_str, tic) -> score
        self.sentiment_lookup = {
            (str(row.date), str(row.tic)): float(row.score)
            for row in sentiment_df.itertuples(index=False)
        }
        self.reward_components = []

    def _get_sentiment_vector(self, date):
        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
        # Vector gồm 1 Cash (vị trí 0 = 0.0) + N cổ phiếu
        sentiment = np.zeros(len(self._tics) + 1, dtype=np.float32)

        for i, tic in enumerate(self._tics, start=1):
            sentiment[i] = self.sentiment_lookup.get((date_str, tic), 0.0)

        return sentiment

    @staticmethod
    def _rank_weights_change(delta_weights):
        return pd.Series(delta_weights).rank(method="average").to_numpy(dtype=np.float32)

    def _calculate_sentiment_rewards(self, weights, previous_weights, sentiment):
        # Bỏ qua Cash ở vị trí 0
        stock_weights = weights[1:]
        stock_previous_weights = previous_weights[1:]
        stock_sentiment = sentiment[1:]

        # R_alpha (Eq. 10): Tích vô hướng giữa trọng số và điểm cảm xúc
        R_alpha = np.sum(stock_weights * stock_sentiment)

        # R_beta (Eq. 11 & 12): Tương tác pairwise rank
        delta_weights = stock_weights - stock_previous_weights
        ranks = self._rank_weights_change(delta_weights)

        sentiment_diff = stock_sentiment[:, None] - stock_sentiment[None, :]
        rank_diff = ranks[:, None] - ranks[None, :]
        R_beta = np.sum(sentiment_diff * rank_diff)

        return float(R_alpha), float(R_beta)

    def step(self, actions):
        result = super().step(actions)

        if self._new_gym_api:
            state, base_reward, terminal, truncated, info = result
        else:
            state, base_reward, terminal, info = result
            truncated = False

        if terminal:
            return result

        # Log return gốc: R = ln(V_t / V_{t-1})
        base_log_return = np.log(
            max(self._asset_memory["final"][-1] / (self._asset_memory["final"][-2] + 1e-8), 1e-8)
        )

        current_weights = np.asarray(self._actions_memory[-1], dtype=np.float32)
        previous_weights = np.asarray(self._final_weights[-2], dtype=np.float32)

        # Lấy ngày tương ứng phiên giao dịch
        date = info.get("end_time", info.get("date", None))
        sentiment = self._get_sentiment_vector(date)

        R_alpha, R_beta = self._calculate_sentiment_rewards(
            weights=current_weights,
            previous_weights=previous_weights,
            sentiment=sentiment
        )

        # Tổng hợp Reward theo chế độ
        if self.reward_mode == "R1":
            reward_unscaled = base_log_return
        elif self.reward_mode == "R2":
            reward_unscaled = base_log_return + self.sentiment_weight * R_alpha
        elif self.reward_mode == "R3":
            reward_unscaled = base_log_return + self.sentiment_weight * R_alpha + self.beta_weight * R_beta

        reward = reward_unscaled * self._reward_scaling
        self._reward = reward
        self._portfolio_reward_memory[-1] = reward_unscaled

        info = dict(info)
        info.update({
            "base_log_return": float(base_log_return),
            "R_alpha": float(R_alpha),
            "R_beta": float(R_beta),
            "reward_unscaled": float(reward_unscaled),
            "portfolio_value": self._portfolio_value,
            "weights": current_weights
        })

        if self._new_gym_api:
            return state, reward, terminal, truncated, info
        return state, reward, terminal, info

# Ghi đè tên lớp để không cần thay đổi bất kỳ code nào khác
PortfolioZeroRewardPOE = PortfolioZeroRewardPOE_Fixed
print("✅ Đã vá thành công lớp PortfolioZeroRewardPOE!")

In [ ]:
# ====================================================================
# CELL 16: KHỞI TẠO LẠI CÁC MÔI TRƯỜNG VỚI LỚP ĐÃ VÁ
# ====================================================================

INITIAL_AMOUNT = 100_000.0
TRANSACTION_FEE = 0.0025
TIME_WINDOW = 30
FEATURES = ["close", "high", "low"]

# 1. Train Environments
env_train_r1 = PortfolioZeroRewardPOE(
    df_train_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R1",
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE, new_gym_api=True
)

env_train_r2 = PortfolioZeroRewardPOE(
    df_train_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R2",
    sentiment_weight=0.05,
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE, new_gym_api=True
)

# R3: Điều chỉnh trọng số cảm xúc để có cùng bậc độ lớn với log-return (~0.01 - 0.03)
env_train_r3 = PortfolioZeroRewardPOE(
    df_train_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R3",
    sentiment_weight=0.01,           # Scale chuẩn đưa R_alpha về khoảng 0.005 - 0.01
    beta_weight=0.0005,              # Scale chuẩn đưa R_beta (vốn có 100 cặp) về ~0.005
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE, new_gym_api=True
)

# 2. Test Environments
env_test_r1 = PortfolioZeroRewardPOE(
    df_test_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R1",
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE, new_gym_api=True
)

env_test_r2 = PortfolioZeroRewardPOE(
    df_test_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R2",
    sentiment_weight=0.05,
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE, new_gym_api=True
)

env_test_r3 = PortfolioZeroRewardPOE(
    df_test_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R3",
    sentiment_weight=0.01, beta_weight=0.0005,
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE, new_gym_api=True
)

print("✅ Đã khởi tạo lại 6 môi trường (R1, R2, R3).")
print(f" - Danh sách ticker ghi nhận trong env: {env_train_r3._tics}")

In [ ]:
# ====================================================================
# BƯỚC 4: PRE-TRAINING TRANSFORMER THEO CÔNG THỨC (13) CỦA BÀI BÁO
# Tác vụ dự báo giá cổ phiếu ngày tiếp theo (D=1) và 5 ngày tiếp theo (D=5)
# ====================================================================

import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Định dạng chuẩn bị dữ liệu tensor từ FinRL DataFrame
def extract_tensors_from_df(df, tickers, features, time_window=30):
    unique_dates = df['date'].unique()
    num_dates = len(unique_dates)
    N = len(tickers)
    M = time_window
    F = len(features)
    
    # Ma trận giá đóng cửa và quan sát trượt
    tensor_obs = np.zeros((num_dates - M, N, M, F), dtype=np.float32)
    
    # Tổ chức bảng pivot theo từng feature
    pivots = [df.pivot(index='date', columns='tic', values=feat)[tickers].values for feat in features]
    # pivots[f] có shape (num_dates, N)
    
    for t in range(num_dates - M):
        for f_idx in range(F):
            # Window slice: (M, N) -> chuyển thành (N, M)
            window_slice = pivots[f_idx][t : t + M, :].T
            # Chuẩn hóa min-max cục bộ theo từng cửa sổ trượt (tránh look-ahead bias)
            min_val = np.min(window_slice, axis=1, keepdims=True)
            max_val = np.max(window_slice, axis=1, keepdims=True) + 1e-8
            normalized_slice = (window_slice - min_val) / (max_val - min_val)
            tensor_obs[t, :, :, f_idx] = normalized_slice
            
    return tensor_obs

print(">>> Đang chuẩn bị tensor quan sát cho Representation Network...")
train_obs_tensor = extract_tensors_from_df(df_train_prices, TARGET_TICKERS, FEATURES, TIME_WINDOW)

class PretrainDataset(Dataset):
    def __init__(self, obs_data, max_d=5, close_idx=0):
        self.obs = obs_data  # Shape: (T, N, M, F)
        self.max_d = max_d
        self.close_idx = close_idx
        self.valid_len = len(obs_data) - max_d

    def __len__(self):
        return self.valid_len

    def __getitem__(self, idx):
        o_enc = self.obs[idx] # (N, M, F)
        # Task D = 1
        o_dec_1 = self.obs[idx + 1, :, -1:, :]
        y_1 = self.obs[idx + 1, :, -1, self.close_idx]
        # Task D = 5
        future_steps = [self.obs[idx + d, :, -1:, :] for d in range(1, 6)]
        o_dec_5 = np.concatenate(future_steps, axis=1) # (N, 5, F)
        y_5 = np.stack([self.obs[idx + d, :, -1, self.close_idx] for d in range(1, 6)], axis=-1) # (N, 5)

        return (
            torch.tensor(o_enc, dtype=torch.float32),
            torch.tensor(o_dec_1, dtype=torch.float32),
            torch.tensor(y_1, dtype=torch.float32),
            torch.tensor(o_dec_5, dtype=torch.float32),
            torch.tensor(y_5, dtype=torch.float32)
        )

pretrain_dataset = PretrainDataset(train_obs_tensor)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=32, shuffle=True)

# ----------------- KIẾN TRÚC MÔ HÌNH TRANSFORMER -----------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class PortfolioTransformer(nn.Module):
    def __init__(self, feature_dim=3, hidden_dim=64, nhead=4, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.enc_input_fc = nn.Linear(feature_dim, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim*2,
            batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dec_input_fc = nn.Linear(feature_dim, hidden_dim)
        self.pos_decoder = PositionalEncoding(hidden_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim*2,
            batch_first=True, dropout=0.1
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.pred_head = nn.Linear(hidden_dim, 1)

    def encode(self, o_enc):
        B, N, M, F = o_enc.shape
        x = self.enc_input_fc(o_enc.view(B * N, M, F))
        x = self.pos_encoder(x)
        s_enc = self.encoder(x)
        return s_enc.view(B, N, M, self.hidden_dim)

    def decode(self, o_dec, s_enc):
        B, N, D, F = o_dec.shape
        M = s_enc.shape[2]
        tgt = self.pos_decoder(self.dec_input_fc(o_dec.view(B * N, D, F)))
        memory = s_enc.view(B * N, M, self.hidden_dim)
        dec_out = self.decoder(tgt=tgt, memory=memory)
        y_hat = self.pred_head(dec_out).squeeze(-1)
        return y_hat.view(B, N, D)

# Tiến hành huấn luyện Pre-train
H_DIM = 64
transformer_pretrain = PortfolioTransformer(feature_dim=len(FEATURES), hidden_dim=H_DIM).to(device)
optimizer_pt = torch.optim.Adam(transformer_pretrain.parameters(), lr=1e-3)

print(">>> Bắt đầu giai đoạn Tiền huấn luyện Representation Network (20 epochs)...")
for epoch in range(1, 21):
    total_loss, l1_sum, l5_sum = 0.0, 0.0, 0.0
    for o_enc, o_dec_1, y_1, o_dec_5, y_5 in pretrain_loader:
        o_enc = o_enc.to(device)
        o_dec_1, y_1 = o_dec_1.to(device), y_1.to(device)
        o_dec_5, y_5 = o_dec_5.to(device), y_5.to(device)

        optimizer_pt.zero_grad()
        s_enc = transformer_pretrain.encode(o_enc)
        
        # Task 1: D=1 (Eq. 13)
        y_hat_1 = transformer_pretrain.decode(o_dec_1, s_enc).squeeze(-1)
        loss_d1 = torch.mean((y_1 - y_hat_1) ** 2)
        
        # Task 2: D=5 (Eq. 13)
        y_hat_5 = transformer_pretrain.decode(o_dec_5, s_enc)
        loss_d5 = torch.mean((y_5 - y_hat_5) ** 2)
        
        loss = loss_d1 + loss_d5
        loss.backward()
        optimizer_pt.step()
        
        total_loss += loss.item()
        l1_sum += loss_d1.item()
        l5_sum += loss_d5.item()
        
    if epoch % 5 == 0 or epoch == 1:
        n_b = len(pretrain_loader)
        print(f"Epoch [{epoch:02d}/20] - Total Loss: {total_loss/n_b:.6f} (L_P_1: {l1_sum/n_b:.6f}, L_P_5: {l5_sum/n_b:.6f})")

# Đóng gói Representation Module h_theta
class RepresentationNetwork(nn.Module):
    def __init__(self, pretrained_model):
        super().__init__()
        self.enc_input_fc = pretrained_model.enc_input_fc
        self.pos_encoder = pretrained_model.pos_encoder
        self.encoder = pretrained_model.encoder
        self.hidden_dim = pretrained_model.hidden_dim

    def forward(self, o_t):
        # Hỗ trợ cả tensor đầu vào từ FinRL (F x N x M) hoặc (N x M x F)
        if o_t.dim() == 3:
            # Nếu FinRL trả về (F, N, M) -> chuyển về (1, N, M, F)
            if o_t.shape[0] == len(FEATURES):
                o_t = o_t.permute(1, 2, 0).unsqueeze(0)
            else:
                o_t = o_t.unsqueeze(0)
        elif o_t.dim() == 4 and o_t.shape[1] == len(FEATURES):
            o_t = o_t.permute(0, 2, 3, 1)

        B, N, M, F = o_t.shape
        x = self.pos_encoder(self.enc_input_fc(o_t.view(B * N, M, F)))
        s_t = self.encoder(x)
        return s_t.view(B, N, M, self.hidden_dim).squeeze(0)

h_theta = RepresentationNetwork(transformer_pretrain).to(device)
print("✅ Representation Network h_theta đã sẵn sàng chuyển giao cho MCTS & RL!")

In [ ]:
# ====================================================================
# BƯỚC 5: XÂY DỰNG PREDICTION NETWORK f_theta VÀ DYNAMICS NETWORK g_theta
# ====================================================================

import torch.nn.functional as F

class PredictionNetwork(nn.Module):
    """
    Prediction Network f_theta:
    Input: s_t in R^(N x M x H)
    Output:
      - a_t (Policy): Vector tỷ trọng trên Simplex (1 Cash + N Stocks) qua Softmax
      - v_t (Value): Điểm số đánh giá trạng thái hiện tại (scalar)
    """
    def __init__(self, num_assets=11, N=10, M=30, H=64, hidden_dim=256):
        super().__init__()
        self.state_flat_dim = N * M * H
        
        self.shared_fc = nn.Sequential(
            nn.Linear(self.state_flat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        # Policy Head -> 11 trọng số
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_assets)
        )
        # Value Head -> scalar
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, s_t):
        if s_t.dim() == 3:
            s_t = s_t.unsqueeze(0)
        B = s_t.shape[0]
        feat = self.shared_fc(s_t.reshape(B, -1))
        
        a_t = F.softmax(self.policy_head(feat), dim=-1)
        v_t = self.value_head(feat)
        return a_t.squeeze(0), v_t.squeeze(0)


class DynamicsNetwork(nn.Module):
    """
    Dynamics Network g_theta (Internal Environment Simulator):
    Input: s_t in R^(N x M x H) và a_t in R^(num_assets)
    Output:
      - r_t: Immediate reward dự đoán (scalar)
      - s_{t+1}: Trạng thái ẩn kế tiếp R^(N x M x H)
    """
    def __init__(self, num_assets=11, N=10, M=30, H=64, hidden_dim=256):
        super().__init__()
        self.N, self.M, self.H = N, M, H
        self.state_flat_dim = N * M * H
        
        self.state_action_fc = nn.Sequential(
            nn.Linear(self.state_flat_dim + num_assets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        # Next State Head với residual connection
        self.next_state_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.state_flat_dim)
        )
        # Immediate Reward Head
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, s_t, a_t):
        if s_t.dim() == 3:
            s_t = s_t.unsqueeze(0)
        if a_t.dim() == 1:
            a_t = a_t.unsqueeze(0)
            
        B = s_t.shape[0]
        s_flat = s_t.reshape(B, -1)
        sa_cat = torch.cat([s_flat, a_t], dim=-1)
        latent = self.state_action_fc(sa_cat)
        
        delta_s = self.next_state_head(latent)
        s_next = (s_flat + delta_s).reshape(B, self.N, self.M, self.H)
        r_t = self.reward_head(latent)
        
        return r_t.squeeze(0), s_next.squeeze(0)

print("✅ Đã thiết lập xong Prediction Network và Dynamics Network.")

In [ ]:
# ====================================================================
# BƯỚC 6: MONTE CARLO TREE SEARCH (MCTS) TRÊN KHÔNG GIAN LIÊN TỤC
# ====================================================================

class OUNoise:
    """Nhiễu Ornstein-Uhlenbeck phục vụ mở rộng hành động liên tục."""
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.size = size
        self.mu = mu * np.ones(self.size)
        self.theta = theta
        self.sigma = sigma
        self.reset()

    def reset(self):
        self.state = np.copy(self.mu)

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.size)
        self.state += dx
        return torch.tensor(self.state, dtype=torch.float32)

class MCTSNode:
    def __init__(self, state, prior_prob=1.0):
        self.state = state
        self.prior_prob = prior_prob
        self.is_expanded = False
        self.children = {}
        self.actions = []
        self.rewards = {}
        self.visit_count = 0
        self.action_visits = {}
        self.action_q_values = {}

    def expand(self, actions, dynamics_net):
        self.actions = actions
        with torch.no_grad():
            for idx, a in enumerate(actions):
                pred_r, s_next = dynamics_net(self.state, a)
                self.rewards[idx] = pred_r.item()
                self.children[idx] = MCTSNode(state=s_next, prior_prob=1.0 / len(actions))
                self.action_visits[idx] = 0
                self.action_q_values[idx] = 0.0
        self.is_expanded = True

    def select_action_idx(self, c_puct=1.25):
        best_score = -float('inf')
        best_idx = 0
        total_visits = sum(self.action_visits.values())

        for idx in range(len(self.actions)):
            n_sa = self.action_visits[idx]
            q_sa = self.action_q_values[idx]
            p_sa = 1.0 / len(self.actions)
            u_score = c_puct * p_sa * (np.sqrt(total_visits) / (1.0 + n_sa))
            score = q_sa + u_score
            if score > best_score:
                best_score = score
                best_idx = idx
        return best_idx

class ContinuousMCTS:
    def __init__(self, pred_net, dyn_net, iota=4, num_simulations=20, gamma=0.99, c_puct=1.25):
        self.pred_net = pred_net
        self.dyn_net = dyn_net
        self.iota = iota
        self.num_simulations = num_simulations
        self.gamma = gamma
        self.c_puct = c_puct
        self.ou_process = OUNoise(size=pred_net.policy_head[-1].out_features)

    def _generate_candidate_actions(self, s_t):
        with torch.no_grad():
            a_base, _ = self.pred_net(s_t)
        candidate_actions = [a_base]
        
        # Thêm iota - 1 hành động chứa nhiễu OU
        for _ in range(self.iota - 1):
            noise = self.ou_process.sample().to(s_t.device)
            perturbed_a = torch.clamp(a_base + noise, min=1e-5)
            perturbed_a = perturbed_a / torch.sum(perturbed_a)
            candidate_actions.append(perturbed_a)
        return candidate_actions

    def search(self, root_state):
        root_node = MCTSNode(state=root_state)
        root_actions = self._generate_candidate_actions(root_state)
        root_node.expand(root_actions, self.dyn_net)

        for _ in range(self.num_simulations):
            node = root_node
            search_path = [node]
            action_path = []

            # Selection
            while node.is_expanded and len(node.children) > 0:
                act_idx = node.select_action_idx(self.c_puct)
                action_path.append(act_idx)
                node = node.children[act_idx]
                search_path.append(node)

            # Expansion
            if not node.is_expanded:
                leaf_actions = self._generate_candidate_actions(node.state)
                node.expand(leaf_actions, self.dyn_net)

            # Evaluation
            with torch.no_grad():
                _, leaf_v = self.pred_net(node.state)
                g_return = leaf_v.item()

            # Backpropagation
            for i in reversed(range(len(action_path))):
                parent_node = search_path[i]
                act_idx = action_path[i]
                reward = parent_node.rewards[act_idx]
                g_return = reward + self.gamma * g_return
                
                parent_node.visit_count += 1
                parent_node.action_visits[act_idx] += 1
                n_count = parent_node.action_visits[act_idx]
                q_old = parent_node.action_q_values[act_idx]
                parent_node.action_q_values[act_idx] = q_old + (g_return - q_old) / n_count

        visits = [root_node.action_visits[i] for i in range(self.iota)]
        best_action_idx = int(np.argmax(visits))
        return root_node.actions[best_action_idx]

print("✅ Module ContinuousMCTS đã được nạp thành công.")

In [ ]:
# ====================================================================
# CELL 20: HÀM HUẤN LUYỆN ALGORITHM 1 (ĐÃ IMPORT RANDOM ĐẦY ĐỦ)
# ====================================================================

import random
import copy
import time
import numpy as np
import torch
import torch.nn.functional as F

class ReplayBuffer:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.buffer = []

    def push(self, trajectory):
        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)
        self.buffer.append(trajectory)

    def sample(self):
        return random.choice(self.buffer)

def compute_joint_loss(pred_a, target_a, pred_v, target_v, pred_r, target_r):
    """
    L_t(theta) = CrossEntropy(a, hat_a) + MSE(v, hat_v) + MSE(r, hat_r) (Eq. 14)
    """
    loss_a = -torch.sum(target_a * torch.log(pred_a + 1e-8))
    loss_v = F.mse_loss(pred_v.view(-1), target_v.view(-1))
    loss_r = F.mse_loss(pred_r.view(-1), target_r.view(-1))
    return loss_a + loss_v + loss_r

def train_portfoliozero_model(
    env,
    h_net,
    f_net,
    g_net,
    model_name="PortfolioZero_R3",
    epochs=35,
    unroll_k=5,
    mcts_sims=20,
    gamma=0.99,
    lr=1e-4,
    l2_reg=1e-4,
    sample_steps=64
):
    print("=" * 75)
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN: {model_name.upper()}")
    print(f" - Số Epochs: {epochs} | Unroll K: {unroll_k} | MCTS Sims: {mcts_sims}")
    print(f" - Tối ưu hóa: Adam (lr={lr}, weight_decay={l2_reg})")
    print("=" * 75)
    
    mcts = ContinuousMCTS(f_net, g_net, iota=4, num_simulations=mcts_sims, gamma=gamma)
    optimizer = torch.optim.Adam(
        list(h_net.parameters()) + list(f_net.parameters()) + list(g_net.parameters()),
        lr=lr, weight_decay=l2_reg
    )
    replay_buffer = ReplayBuffer(capacity=10)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        obs_raw, _ = env.reset()
        h_net.eval(); f_net.eval(); g_net.eval()
        
        trajectory = {'obs': [], 'actions': [], 'rewards': []}
        done = False
        
        # 1. Giai đoạn thực thi (Execution Phase)
        while not done:
            obs_t = torch.tensor(obs_raw, dtype=torch.float32).to(device)
            with torch.no_grad():
                s_t = h_net(obs_t)
            
            # MCTS tìm kiếm action mục tiêu
            hat_a_t = mcts.search(s_t)
            act_np = hat_a_t.detach().cpu().numpy()
            
            # Tương tác với môi trường FinRL
            next_obs, reward, terminated, truncated, _ = env.step(act_np)
            
            trajectory['obs'].append(obs_t.cpu())
            trajectory['actions'].append(hat_a_t.cpu())
            trajectory['rewards'].append(reward)
            
            obs_raw = next_obs
            done = terminated or truncated

        # Tính toán giá trị chiết khấu thực tế hat{v}_t (Dòng 19)
        T = len(trajectory['rewards'])
        returns = np.zeros(T, dtype=np.float32)
        running_ret = 0.0
        for i in reversed(range(T)):
            running_ret = trajectory['rewards'][i] + gamma * running_ret
            returns[i] = running_ret
        trajectory['returns'] = [torch.tensor(r, dtype=torch.float32) for r in returns]
        replay_buffer.push(trajectory)
        
        cum_ret = np.sum(trajectory['rewards'])

        # 2. Giai đoạn tối ưu gradient (Training Phase - K steps unrolling)
        h_net.train(); f_net.train(); g_net.train()
        trace = replay_buffer.sample()
        T_trace = len(trace['rewards'])
        valid_indices = list(range(0, T_trace - unroll_k))
        sample_idx = np.random.choice(valid_indices, size=min(sample_steps, len(valid_indices)), replace=False)
        
        epoch_loss = 0.0
        for t_start in sample_idx:
            optimizer.zero_grad()
            L_theta = 0.0
            
            for k in range(unroll_k):
                idx = t_start + k
                o_k = trace['obs'][idx].to(device)
                hat_a_k = trace['actions'][idx].to(device)
                hat_r_k = torch.tensor(trace['rewards'][idx], dtype=torch.float32).to(device)
                hat_v_k = trace['returns'][idx].to(device)
                
                s_k = h_net(o_k)
                a_k, v_k = f_net(s_k)
                pred_r_k, _ = g_net(s_k, hat_a_k)
                
                step_loss = compute_joint_loss(a_k, hat_a_k, v_k, hat_v_k, pred_r_k, hat_r_k)
                L_theta += step_loss
                
            L_theta.backward()
            optimizer.step()
            epoch_loss += L_theta.item()
            
        elapsed = time.time() - t0
        print(f"[{model_name}] Epoch {epoch:02d}/{epochs:02d} ({elapsed:.1f}s) | Cumulative Reward: {cum_ret:.4f} | Loss: {epoch_loss/len(sample_idx):.4f}")
        
    return h_net, f_net, g_net

print("✅ Đã cập nhật xong Cell 20 (Sửa lỗi random.choice).")

In [ ]:
# ====================================================================
# CELL 21: BẬT / TẮT LỰA CHỌN HUẤN LUYỆN (R1, R2, R3)
# ====================================================================

# CẤU HÌNH BẬT / TẮT CHẠY MODEL (Tùy chọn theo ý muốn)
RUN_R1 = True   # Đặt True nếu muốn chạy R1
RUN_R2 = True  # Đặt True nếu muốn chạy R2
RUN_R3 = True    # Bật chạy mô hình hoàn chỉnh R3 (sát bài báo nhất)

EPOCHS_TO_TRAIN = 35  # Số epoch huấn luyện (Khuyên dùng 30-40 epochs để hội tụ tốt)
NUM_ASSETS = len(TARGET_TICKERS) + 1  # 1 Cash + 10 Cổ phiếu = 11

# --- 1. Huấn luyện PortfolioZero_R1 ---
if RUN_R1:
    h_r1 = copy.deepcopy(h_theta)
    f_r1 = PredictionNetwork(num_assets=NUM_ASSETS, N=len(TARGET_TICKERS)).to(device)
    g_r1 = DynamicsNetwork(num_assets=NUM_ASSETS, N=len(TARGET_TICKERS)).to(device)
    h_r1, f_r1, g_r1 = train_portfoliozero_model(
        env_train_r1, h_r1, f_r1, g_r1, 
        model_name="PortfolioZero_R1", epochs=EPOCHS_TO_TRAIN
    )
else:
    h_r1, f_r1, g_r1 = None, None, None
    print("⏭️ Bỏ qua huấn luyện PortfolioZero_R1.")

# --- 2. Huấn luyện PortfolioZero_R2 ---
if RUN_R2:
    h_r2 = copy.deepcopy(h_theta)
    f_r2 = PredictionNetwork(num_assets=NUM_ASSETS, N=len(TARGET_TICKERS)).to(device)
    g_r2 = DynamicsNetwork(num_assets=NUM_ASSETS, N=len(TARGET_TICKERS)).to(device)
    h_r2, f_r2, g_r2 = train_portfoliozero_model(
        env_train_r2, h_r2, f_r2, g_r2, 
        model_name="PortfolioZero_R2", epochs=EPOCHS_TO_TRAIN
    )
else:
    h_r2, f_r2, g_r2 = None, None, None
    print("⏭️ Bỏ qua huấn luyện PortfolioZero_R2.")

# --- 3. Huấn luyện PortfolioZero_R3 (Mô hình chính thức chuẩn bài báo) ---
if RUN_R3:
    h_r3 = copy.deepcopy(h_theta)
    f_r3 = PredictionNetwork(num_assets=NUM_ASSETS, N=len(TARGET_TICKERS)).to(device)
    g_r3 = DynamicsNetwork(num_assets=NUM_ASSETS, N=len(TARGET_TICKERS)).to(device)
    h_r3, f_r3, g_r3 = train_portfoliozero_model(
        env_train_r3, h_r3, f_r3, g_r3, 
        model_name="PortfolioZero_R3", epochs=EPOCHS_TO_TRAIN
    )
else:
    h_r3, f_r3, g_r3 = None, None, None
    print("⏭️ Bỏ qua huấn luyện PortfolioZero_R3.")

print("\n🎉 Hoàn thành lượt chạy huấn luyện!")

In [ ]:
# ====================================================================
# CELL 22: ĐÁNH GIÁ CHUẨN TÀI CHÍNH (SỬA CÔNG THỨC SHARPE TỪ GIÁ TRỊ TÀI SẢN)
# ====================================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

def evaluate_portfolio_metrics(portfolio_values, annual_days=252):
    p_arr = np.array(portfolio_values, dtype=np.float64)
    p0 = p_arr[0]
    pT = p_arr[-1]
    T_steps = len(p_arr) - 1
    
    # 1. Logarithmic Accumulation of Portfolio Value (LAPV - Eq. 15)
    lapv = float(np.log(max(pT / p0, 1e-8)))
    
    # 2. Annualized Rate of Return (ARR - Eq. 16)
    arr = float((pT / p0) ** (annual_days / T_steps) - 1.0) if T_steps > 0 else 0.0
    
    # 3. CHUẨN XÁC: Tính Daily Return thực tế từ Portfolio Value: r_t = (p_t - p_{t-1}) / p_{t-1}
    daily_pct_returns = np.diff(p_arr) / p_arr[:-1]
    std_r = np.std(daily_pct_returns)
    mean_r = np.mean(daily_pct_returns)
    
    # Giả định Risk-free rate Rf = 2%/năm (~ 0.02 / 252 mỗi ngày)
    rf_daily = 0.02 / annual_days
    sr = float(((mean_r - rf_daily) / (std_r + 1e-8)) * np.sqrt(annual_days)) if std_r > 0 else 0.0
    
    # 4. Maximum Drawdown (MDD - Eq. 18)
    peak = np.maximum.accumulate(p_arr)
    drawdowns = (peak - p_arr) / peak
    mdd = float(np.max(drawdowns)) if len(drawdowns) > 0 else 0.0
    
    # 5. Calmar Ratio (CR - Eq. 19)
    cr = float((arr / mdd) * 100) if mdd > 1e-6 else 0.0
    
    return {
        'LAPV': round(lapv, 4),
        'ARR (%)': round(arr * 100, 4),
        'SR': round(sr, 4),
        'MDD': round(mdd, 4),
        'CR (%)': round(cr, 4)
    }

def run_test_inference(env, h_net, f_net):
    h_net.eval(); f_net.eval()
    obs_raw, _ = env.reset()
    done = False
    
    init_val = getattr(env, '_portfolio_value', getattr(env, '_initial_amount', 100000.0))
    pv_history = [init_val]
    
    with torch.no_grad():
        while not done:
            obs_t = torch.tensor(obs_raw, dtype=torch.float32).to(device)
            s_t = h_net(obs_t)
            a_t, _ = f_net(s_t)
            act_np = a_t.cpu().numpy()
            
            next_obs, reward, terminated, truncated, info = env.step(act_np)
            pv_history.append(info.get('portfolio_value', env._portfolio_value))
            obs_raw = next_obs
            done = terminated or truncated
            
    return pv_history

def run_equal_weight_baseline(env):
    env.reset()
    done = False
    eq_action = np.ones(env.action_space.shape[0]) / env.action_space.shape[0]
    
    init_val = getattr(env, '_portfolio_value', getattr(env, '_initial_amount', 100000.0))
    pv_history = [init_val]
    
    while not done:
        _, reward, terminated, truncated, info = env.step(eq_action)
        pv_history.append(info.get('portfolio_value', env._portfolio_value))
        done = terminated or truncated
        
    return pv_history

print(">>> Đang chạy Backtest Out-of-Sample 2023...")

# 1. Baseline Equal Weight
pv_ew = run_equal_weight_baseline(env_test_r3)
summary_list = [{"Strategy": "Equal Weight (Benchmark)", **evaluate_portfolio_metrics(pv_ew)}]

# 2. PortfolioZero_R3
if 'h_r3' in globals() and h_r3 is not None and f_r3 is not None:
    pv_r3 = run_test_inference(env_test_r3, h_r3, f_r3)
    summary_list.append({"Strategy": "PortfolioZero_R3", **evaluate_portfolio_metrics(pv_r3)})

# Bảng so sánh 5 chỉ số tài chính chuẩn
df_summary = pd.DataFrame(summary_list)
print("\n" + "=" * 80)
print("     BẢNG SO SÁNH HIỆU QUẢ ĐẦU TƯ TRÊN 10 MÃ CỔ PHIẾU (OUT-OF-SAMPLE 2023)")
print("=" * 80)
print(df_summary.to_string(index=False))
print("=" * 80)

# Vẽ đồ thị đường cong tăng trưởng tích lũy logarit (LAPV)
plt.figure(figsize=(12, 6))
plt.plot(np.log(np.array(pv_ew) / pv_ew[0]), label='Equal Weight Baseline', color='gray', linestyle='--', linewidth=1.5)
if 'pv_r3' in locals():
    plt.plot(np.log(np.array(pv_r3) / pv_r3[0]), label='PortfolioZero_R3 (FinBERT Sentiment)', color='red', linewidth=2.2)

plt.title("Đường cong tích lũy giá trị tài sản logarit (LAPV) - 10 US Stocks 2023", fontsize=13, fontweight='bold')
plt.xlabel("Ngày giao dịch (Trading Days)", fontsize=11)
plt.ylabel("Log Portfolio Value (LAPV)", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("✅ Đã hoàn tất đánh giá chuẩn xác!")

In [ ]:
# ====================================================================
# CELL LƯU & TẢI CHECKPOINT CHO MÔ HÌNH PORTFOLIOZERO_R3
# ====================================================================

import os
import torch

# 1. Định nghĩa thư mục và tên file checkpoint
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_portfoliozero_r3.pth")

def save_portfoliozero_checkpoint(
    path=CHECKPOINT_PATH,
    h_net=None,
    f_net=None,
    g_net=None,
    metrics=None,
    epoch=None,
    extra_info=None
):
    """Lưu toàn bộ trạng thái mạng và thông số thực nghiệm vào file .pth"""
    if h_net is None or f_net is None or g_net is None:
        raise ValueError("Cần cung cấp đầy đủ cả 3 mạng: h_net, f_net, g_net để lưu checkpoint!")
        
    checkpoint = {
        'model_name': 'PortfolioZero_R3',
        'epoch': epoch,
        'h_net_state_dict': h_net.state_dict(),
        'f_net_state_dict': f_net.state_dict(),
        'g_net_state_dict': g_net.state_dict(),
        'target_tickers': TARGET_TICKERS,
        'time_window': TIME_WINDOW,
        'features': FEATURES,
        'hidden_dim': H_DIM,
        'metrics': metrics,
        'extra_info': extra_info
    }
    
    torch.save(checkpoint, path)
    file_size_mb = os.path.getsize(path) / (1024 * 1024)
    print("=" * 70)
    print(f"💾 ĐÃ LƯU CHECKPOINT THÀNH CÔNG!")
    print(f" - Đường dẫn lưu: {path}")
    print(f" - Dung lượng file: {file_size_mb:.2f} MB")
    if metrics:
        print(f" - Hiệu năng lưu kèm (Metrics):")
        for k, v in metrics.items():
            print(f"    + {k}: {v}")
    print("=" * 70)


def load_portfoliozero_checkpoint(path=CHECKPOINT_PATH, device=device):
    """Tải lại trọng số 3 mạng từ file checkpoint đã lưu để suy luận hoặc train tiếp"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file checkpoint tại: {path}")
        
    print(f">>> Đang tải checkpoint từ {path}...")
    checkpoint = torch.load(path, map_location=device)
    
    # Khởi tạo lại khung mạng nếu cần
    num_assets = len(checkpoint['target_tickers']) + 1
    N = len(checkpoint['target_tickers'])
    M = checkpoint['time_window']
    H = checkpoint['hidden_dim']
    
    # Tạo instance mới
    loaded_transformer = PortfolioTransformer(feature_dim=len(checkpoint['features']), hidden_dim=H).to(device)
    loaded_h = RepresentationNetwork(loaded_transformer).to(device)
    loaded_f = PredictionNetwork(num_assets=num_assets, N=N, M=M, H=H).to(device)
    loaded_g = DynamicsNetwork(num_assets=num_assets, N=N, M=M, H=H).to(device)
    
    # Nạp state dict
    loaded_h.load_state_dict(checkpoint['h_net_state_dict'])
    loaded_f.load_state_dict(checkpoint['f_net_state_dict'])
    loaded_g.load_state_dict(checkpoint['g_net_state_dict'])
    
    loaded_h.eval()
    loaded_f.eval()
    loaded_g.eval()
    
    print("✅ Đã khôi phục hoàn chỉnh 3 mạng: h_theta, f_theta, g_theta!")
    if checkpoint.get('metrics'):
        print(f" - Thông số hiệu năng đã lưu: {checkpoint['metrics']}")
        
    return loaded_h, loaded_f, loaded_g, checkpoint


# ====================================================================
# TIẾN HÀNH LƯU MÔ HÌNH VỪA HUẤN LUYỆN
# ====================================================================
if 'h_r3' in globals() and h_r3 is not None:
    # Lấy metric của R3 vừa đánh giá ở Cell 22 (nếu có)
    r3_metrics = None
    if 'metrics_r3' in globals():
        r3_metrics = metrics_r3
    elif 'summary_list' in globals():
        for item in summary_list:
            if "R3" in item.get("Strategy", ""):
                r3_metrics = item
                break

    save_portfoliozero_checkpoint(
        path=CHECKPOINT_PATH,
        h_net=h_r3,
        f_net=f_r3,
        g_net=g_r3,
        metrics=r3_metrics,
        epoch=EPOCHS_TO_TRAIN if 'EPOCHS_TO_TRAIN' in globals() else 30,
        extra_info="Huấn luyện trên 10 mã US Stocks với FinBERT Sentiment R3"
    )
else:
    print("⚠️ Chưa tìm thấy biến h_r3 trong bộ nhớ. Hãy chạy xong Cell 21 trước khi lưu.")

In [ ]:
# ====================================================================
# CELL A: HUẤN LUYỆN TÁC TỬ FinRL PG (KHỚP TIME_WINDOW = 30)
# ====================================================================

import torch
import numpy as np
from finrl.agents.portfolio_optimization.models import DRLAgent

print("=" * 75)
print("🚀 BẮT ĐẦU HUẤN LUYỆN TÁC TỬ FinRL POLICY GRADIENT (EIIE POLICY)")
print("=" * 75)

# 1. Khởi tạo môi trường dành riêng cho FinRL (new_gym_api=False)
env_train_finrl = PortfolioZeroRewardPOE(
    df_train_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R1",
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE,
    new_gym_api=False   # FinRL yêu cầu Gym API cũ
)

# 2. Khởi tạo tác tử DRLAgent
finrl_agent = DRLAgent(env=env_train_finrl)

# 3. TRUYỀN TIME_WINDOW=30 VÀO POLICY_KWARGS ĐỂ TÍCH CHẬP Conv2D VỪA VẶN
# Tránh lỗi Kernel size (1 x 48) > Input size (1 x 28)
model_pg = finrl_agent.get_model(
    model_name="pg",
    device=device,
    model_kwargs={"lr": 1e-4},
    policy_kwargs={"time_window": TIME_WINDOW}
)

# 4. Huấn luyện trong 30 episodes
EPISODES = 35
print(f">>> Đang huấn luyện FinRL PG trong {EPISODES} episodes...")
trained_finrl_model = finrl_agent.train_model(
    model=model_pg,
    episodes=EPISODES
)

print("\n✅ Đã huấn luyện thành công tác tử FinRL Policy Gradient!")

In [ ]:
# ====================================================================
# CELL B: SO SÁNH HIỆU NĂNG TỔNG THỂ & VẼ ĐỒ THỊ CHUẨN BÀI BÁO PORTFOLIOZERO
# Đối chiếu: Equal Weight vs FinRL PG vs PortfolioZero_R3
# ====================================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# 1. Môi trường Test cho FinRL (new_gym_api=False)
env_test_finrl = PortfolioZeroRewardPOE(
    df_test_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R1",
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE,
    new_gym_api=False
)

def run_finrl_pg_inference(env, model):
    """Suy luận bằng mô hình FinRL PG trên tập Test"""
    obs = env.reset()
    done = False
    
    init_val = getattr(env, '_portfolio_value', getattr(env, '_initial_amount', 100000.0))
    pv_history = [init_val]
    
    # FinRL Policy cần biết previous action (bắt đầu bằng 100% Cash)
    last_action = np.zeros(env.action_space.shape[0], dtype=np.float32)
    last_action[0] = 1.0

    with torch.no_grad():
        while not done:
            obs_batch = np.expand_dims(obs, axis=0)
            last_action_batch = np.expand_dims(last_action, axis=0)
            
            # Predict action qua policy EIIE của FinRL
            if hasattr(model, 'train_policy'):
                action = model.train_policy(obs_batch, last_action_batch)
            elif hasattr(model, 'policy'):
                action = model.policy(obs_batch, last_action_batch)
            else:
                action = model(obs_batch, last_action_batch)
                
            if isinstance(action, torch.Tensor):
                action = action.cpu().numpy()
                
            action = np.squeeze(action)
            action = np.clip(action, 0.0, None)
            act_sum = np.sum(action)
            if act_sum > 0:
                action = action / act_sum
            else:
                action = np.ones(env.action_space.shape[0]) / env.action_space.shape[0]
                
            last_action = action
            
            # Step môi trường
            step_result = env.step(action)
            if len(step_result) == 5:
                obs, reward, terminated, truncated, info = step_result
                done = terminated or truncated
            else:
                obs, reward, done, info = step_result
                
            pv_history.append(info.get('portfolio_value', env._portfolio_value))
            
    return pv_history

print(">>> Đang chạy Backtest Out-of-Sample 2023 cho cả 3 chiến lược...")

# 1. Lấy dữ liệu Baseline Equal Weight
pv_ew = run_equal_weight_baseline(env_test_r3)
metrics_ew = evaluate_portfolio_metrics(pv_ew)

# 2. Lấy dữ liệu FinRL PG
pv_finrl = run_finrl_pg_inference(env_test_finrl, trained_finrl_model)
metrics_finrl = evaluate_portfolio_metrics(pv_finrl)

# 3. Lấy dữ liệu PortfolioZero_R3
if 'h_r3' in globals() and h_r3 is not None and f_r3 is not None:
    pv_r3 = run_test_inference(env_test_r3, h_r3, f_r3)
    metrics_r3 = evaluate_portfolio_metrics(pv_r3)
else:
    loaded_h, loaded_f, _, _ = load_portfoliozero_checkpoint()
    pv_r3 = run_test_inference(env_test_r3, loaded_h, loaded_f)
    metrics_r3 = evaluate_portfolio_metrics(pv_r3)

# ====================================================================
# BẢNG TỔNG HỢP HIỆU NĂNG TƯƠNG TỰ TABLE 2, 3, 4 TRONG BÀI BÁO
# ====================================================================
comparison_df = pd.DataFrame([
    {"Strategy": "Equal Weight (Benchmark)", **metrics_ew},
    {"Strategy": "FinRL PG (Policy Gradient / EIIE)", **metrics_finrl},
    {"Strategy": "PortfolioZero_R3 (MCTS + FinBERT)", **metrics_r3}
])

print("\n" + "=" * 85)
print("     BẢNG ĐỐI CHIẾU HIỆU NĂNG ĐẦU TƯ TẬP 10 CỔ PHIẾU (OUT-OF-SAMPLE 2023)")
print("=" * 85)
print(comparison_df.to_string(index=False))
print("=" * 85)

# ====================================================================
# VẼ ĐỒ THỊ LAPV CURVE (FIGURE 3, 4, 5 THEO CHUẨN BÀI BÁO)
# ====================================================================
plt.figure(figsize=(14, 6), dpi=150)

plot_steps = range(len(pv_ew))

lapv_ew    = np.log(np.array(pv_ew) / pv_ew[0])
lapv_finrl = np.log(np.array(pv_finrl) / pv_finrl[0])
lapv_r3    = np.log(np.array(pv_r3) / pv_r3[0])

plt.plot(plot_steps, lapv_ew,    label='Equal Weight (Baseline)', color='#7f7f7f', linestyle='--', linewidth=1.6)
plt.plot(plot_steps, lapv_finrl, label='FinRL PG (Classic DRL / EIIE)', color='#1f77b4', linestyle='-.', linewidth=1.8)
plt.plot(plot_steps, lapv_r3,    label='PortfolioZero_R3 (MCTS + FinBERT)', color='#d62728', linewidth=2.4)

plt.title("Logarithmic Accumulation of Portfolio Value (LAPV) - Out-of-Sample 2023\n10 US Equities (GILD, NVDA, TSLA, MRK, KO, AAPL, GS, WMT, FDX, AMT)", 
          fontsize=13, fontweight='bold', pad=12)

plt.xlabel("Trading Days (Jan 2023 - Dec 2023)", fontsize=11, fontweight='medium')
plt.ylabel("LAPV: ln(P_t / P_0)", fontsize=11, fontweight='medium')

plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.legend(fontsize=11, loc='upper left', frameon=True, shadow=True)
plt.tight_layout()
plt.show()

print("🎉 Hoàn tất so sánh đối đầu giữa FinRL PG và PortfolioZero_R3!")

In [ ]:
# ====================================================================
# CELL VẼ BIỂU ĐỒ CHUẨN BÀI BÁO (HIỂN THỊ RÕ TRỤC X/Y VÀ NGÀY THÁNG CỤ THỂ)
# ====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Trích xuất danh sách ngày giao dịch thực tế từ dữ liệu Test
test_dates = df_test_prices['date'].unique()
test_dates = sorted(test_dates)

# Cắt ngày cho khớp chính xác với độ dài chuỗi giá trị tài sản (pv_ew)
num_points = len(pv_ew)
if len(test_dates) >= num_points:
    date_series = pd.to_datetime(test_dates[:num_points])
else:
    # Nếu thiếu 1 điểm (do ngày khởi tạo ban đầu t0), bổ sung ngày liền trước
    dt_start = pd.to_datetime(test_dates[0]) - pd.Timedelta(days=1)
    full_dates = [dt_start] + list(pd.to_datetime(test_dates))
    date_series = pd.to_datetime(full_dates[:num_points])

# 2. Tính giá trị logarit tích lũy (LAPV)
lapv_ew    = np.log(np.array(pv_ew) / pv_ew[0])
lapv_finrl = np.log(np.array(pv_finrl) / pv_finrl[0])
lapv_r3    = np.log(np.array(pv_r3) / pv_r3[0])

# 3. Tạo khung đồ thị với độ sắc nét cao
fig, ax = plt.subplots(figsize=(14, 6.5), dpi=150)

# Đường Zero-line (Mốc 0.0 hòa vốn)
ax.axhline(0, color='black', linestyle='-', linewidth=0.9, alpha=0.7)

# Vẽ 3 đường chiến lược
ax.plot(date_series, lapv_ew,    label='Equal Weight (Baseline)', color='#7f7f7f', linestyle='--', linewidth=1.6)
ax.plot(date_series, lapv_finrl, label='FinRL PG (Classic DRL / EIIE)', color='#1f77b4', linestyle='-.', linewidth=1.8)
ax.plot(date_series, lapv_r3,    label='PortfolioZero_R3 (MCTS + FinBERT)', color='#d62728', linestyle='-', linewidth=2.3)

# ----------------- ĐỊNH DẠNG TRỤC TỌA ĐỘ RÕ RÀNG -----------------
# Làm đậm viền trục X và trục Y
for spine in ['left', 'bottom', 'top', 'right']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_color('black')
    ax.spines[spine].set_linewidth(1.2)

# Hiển thị vạch chia độ (ticks) rõ ràng ở cả trục X và Y
ax.tick_params(axis='both', which='major', direction='inout', length=6, width=1.2, colors='black', labelsize=10)

# Định dạng hiển thị ngày tháng cụ thể trên trục X (Format: YYYY-MM-DD)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))  # Mỗi tháng hiển thị 1 mốc
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate(rotation=30, ha='right')  # Xoay nghiêng 30 độ để các ngày không đè lên nhau

# Tiêu đề và nhãn
ax.set_title("Logarithmic Accumulation of Portfolio Value (LAPV) - Out-of-Sample 2023\n10 US Equities (GILD, NVDA, TSLA, MRK, KO, AAPL, GS, WMT, FDX, AMT)", 
             fontsize=13, fontweight='bold', pad=14, color='black')

ax.set_xlabel("Date (Year-Month-Day)", fontsize=11, fontweight='bold', labelpad=8, color='black')
ax.set_ylabel("LAPV: ln(P_t / P_0)", fontsize=11, fontweight='bold', labelpad=8, color='black')

# Giới hạn trục ngày hiển thị sát với dữ liệu
ax.set_xlim(date_series[0], date_series[-1])

# Lưới mờ hỗ trợ quan sát
ax.grid(True, linestyle=':', alpha=0.5, color='gray')

# Legend vị trí góc trên bên trái nổi bật
ax.legend(fontsize=10.5, loc='upper left', frameon=True, facecolor='white', framealpha=0.95, edgecolor='black')

plt.tight_layout()
plt.show()

print("✅ Đã cập nhật xong biểu đồ với trục X là ngày tháng thực tế và trục X/Y rõ nét!")

In [ ]:
# ====================================================================
# CELL B: SO SÁNH TOÀN DIỆN (EQUAL WEIGHT, FINRL PG, R1, R2, R3)
# Hiển thị trục X ngày tháng cụ thể, viền trục X/Y rõ nét chuẩn bài báo
# ====================================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Khởi tạo môi trường Test cho FinRL PG (new_gym_api=False)
env_test_finrl = PortfolioZeroRewardPOE(
    df_test_prices, INITIAL_AMOUNT,
    sentiment_df=df_sentiment, reward_mode="R1",
    time_window=TIME_WINDOW, features=FEATURES,
    comission_fee_pct=TRANSACTION_FEE,
    new_gym_api=False
)

def run_finrl_pg_inference(env, model):
    """Suy luận bằng mô hình FinRL PG trên tập Test"""
    obs = env.reset()
    done = False
    
    init_val = getattr(env, '_portfolio_value', getattr(env, '_initial_amount', 100000.0))
    pv_history = [init_val]
    
    last_action = np.zeros(env.action_space.shape[0], dtype=np.float32)
    last_action[0] = 1.0

    with torch.no_grad():
        while not done:
            obs_batch = np.expand_dims(obs, axis=0)
            last_action_batch = np.expand_dims(last_action, axis=0)
            
            if hasattr(model, 'train_policy'):
                action = model.train_policy(obs_batch, last_action_batch)
            elif hasattr(model, 'policy'):
                action = model.policy(obs_batch, last_action_batch)
            else:
                action = model(obs_batch, last_action_batch)
                
            if isinstance(action, torch.Tensor):
                action = action.cpu().numpy()
                
            action = np.squeeze(action)
            action = np.clip(action, 0.0, None)
            act_sum = np.sum(action)
            if act_sum > 0:
                action = action / act_sum
            else:
                action = np.ones(env.action_space.shape[0]) / env.action_space.shape[0]
                
            last_action = action
            
            step_result = env.step(action)
            if len(step_result) == 5:
                obs, reward, terminated, truncated, info = step_result
                done = terminated or truncated
            else:
                obs, reward, done, info = step_result
                
            pv_history.append(info.get('portfolio_value', env._portfolio_value))
            
    return pv_history

print(">>> Đang chạy Backtest Out-of-Sample 2023 cho tất cả các chiến lược...")

summary_list = []

# --- 1. Baseline Equal Weight ---
pv_ew = run_equal_weight_baseline(env_test_r3)
metrics_ew = evaluate_portfolio_metrics(pv_ew)
summary_list.append({"Strategy": "Equal Weight (Benchmark)", **metrics_ew})

# --- 2. FinRL PG (Policy Gradient / EIIE) ---
if 'trained_finrl_model' in globals() and trained_finrl_model is not None:
    pv_finrl = run_finrl_pg_inference(env_test_finrl, trained_finrl_model)
    metrics_finrl = evaluate_portfolio_metrics(pv_finrl)
    summary_list.append({"Strategy": "FinRL PG (Classic DRL / EIIE)", **metrics_finrl})
else:
    pv_finrl = None

# --- 3. PortfolioZero_R1 ---
if 'h_r1' in globals() and h_r1 is not None and f_r1 is not None:
    pv_r1 = run_test_inference(env_test_r1, h_r1, f_r1)
    metrics_r1 = evaluate_portfolio_metrics(pv_r1)
    summary_list.append({"Strategy": "PortfolioZero_R1 (Base Log Return)", **metrics_r1})
else:
    pv_r1 = None

# --- 4. PortfolioZero_R2 ---
if 'h_r2' in globals() and h_r2 is not None and f_r2 is not None:
    pv_r2 = run_test_inference(env_test_r2, h_r2, f_r2)
    metrics_r2 = evaluate_portfolio_metrics(pv_r2)
    summary_list.append({"Strategy": "PortfolioZero_R2 (+ R_alpha Sentiment)", **metrics_r2})
else:
    pv_r2 = None

# --- 5. PortfolioZero_R3 ---
if 'h_r3' in globals() and h_r3 is not None and f_r3 is not None:
    pv_r3 = run_test_inference(env_test_r3, h_r3, f_r3)
    metrics_r3 = evaluate_portfolio_metrics(pv_r3)
    summary_list.append({"Strategy": "PortfolioZero_R3 (+ R_alpha + R_beta)", **metrics_r3})
else:
    loaded_h, loaded_f, _, _ = load_portfoliozero_checkpoint()
    pv_r3 = run_test_inference(env_test_r3, loaded_h, loaded_f)
    metrics_r3 = evaluate_portfolio_metrics(pv_r3)
    summary_list.append({"Strategy": "PortfolioZero_R3 (+ R_alpha + R_beta)", **metrics_r3})

# ====================================================================
# BẢNG TỔNG HỢP HIỆU NĂNG TƯƠNG TỰ TABLE 2, 3, 4 TRONG BÀI BÁO
# ====================================================================
comparison_df = pd.DataFrame(summary_list)
print("\n" + "=" * 90)
print("          BẢNG ĐỐI CHIẾU CÁC CHIẾN LƯỢC ĐẦU TƯ TRÊN 10 CỔ PHIẾU (OUT-OF-SAMPLE 2023)")
print("=" * 90)
print(comparison_df.to_string(index=False))
print("=" * 90)

# ====================================================================
# VẼ ĐỒ THỊ ĐA ĐƯỜNG VỚI TRỤC NGÀY THÁNG VÀ VIỀN TỌA ĐỘ RÕ RÀNG
# ====================================================================
# Xử lý chuỗi ngày tháng thực tế từ df_test_prices
test_dates = sorted(df_test_prices['date'].unique())
num_points = len(pv_ew)
if len(test_dates) >= num_points:
    date_series = pd.to_datetime(test_dates[:num_points])
else:
    dt_start = pd.to_datetime(test_dates[0]) - pd.Timedelta(days=1)
    full_dates = [dt_start] + list(pd.to_datetime(test_dates))
    date_series = pd.to_datetime(full_dates[:num_points])

fig, ax = plt.subplots(figsize=(14, 6.8), dpi=150)

# 1. Đường mốc 0.0 (Hòa vốn)
ax.axhline(0, color='black', linestyle='-', linewidth=1.0, alpha=0.7)

# 2. Vẽ Equal Weight
lapv_ew = np.log(np.array(pv_ew) / pv_ew[0])
ax.plot(date_series, lapv_ew, label='Equal Weight (Baseline)', color='#7f7f7f', linestyle='--', linewidth=1.6)

# 3. Vẽ FinRL PG (nếu có)
if pv_finrl is not None:
    lapv_finrl = np.log(np.array(pv_finrl) / pv_finrl[0])
    ax.plot(date_series, lapv_finrl, label='FinRL PG (Classic DRL / EIIE)', color='#1f77b4', linestyle='-.', linewidth=1.8)

# 4. Vẽ PortfolioZero_R1 (nếu có)
if pv_r1 is not None:
    lapv_r1 = np.log(np.array(pv_r1) / pv_r1[0])
    ax.plot(date_series, lapv_r1, label='PortfolioZero_R1 (Base Log Return)', color='#2ca02c', linestyle=':', linewidth=1.8)

# 5. Vẽ PortfolioZero_R2 (nếu có)
if pv_r2 is not None:
    lapv_r2 = np.log(np.array(pv_r2) / pv_r2[0])
    ax.plot(date_series, lapv_r2, label='PortfolioZero_R2 (+ R_alpha Sentiment)', color='#ff7f0e', linestyle='-.', linewidth=2.0)

# 6. Vẽ PortfolioZero_R3 (Mô hình chính)
if pv_r3 is not None:
    lapv_r3 = np.log(np.array(pv_r3) / pv_r3[0])
    ax.plot(date_series, lapv_r3, label='PortfolioZero_R3 (+ R_alpha + R_beta)', color='#d62728', linestyle='-', linewidth=2.5)

# ----------------- ĐỊNH DẠNG KHUNG VIỀN VÀ TRỤC TỌA ĐỘ RÕ RÀNG -----------------
for spine in ['left', 'bottom', 'top', 'right']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_color('black')
    ax.spines[spine].set_linewidth(1.2)

ax.tick_params(axis='both', which='major', direction='inout', length=6, width=1.2, colors='black', labelsize=10)

# Định dạng ngày tháng trục X: YYYY-MM-DD
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate(rotation=30, ha='right')

ax.set_title("Logarithmic Accumulation of Portfolio Value (LAPV) - Out-of-Sample 2023\n10 US Equities (GILD, NVDA, TSLA, MRK, KO, AAPL, GS, WMT, FDX, AMT)", 
             fontsize=13, fontweight='bold', pad=14, color='black')

ax.set_xlabel("Date (Year-Month-Day)", fontsize=11, fontweight='bold', labelpad=8, color='black')
ax.set_ylabel("LAPV: ln(P_t / P_0)", fontsize=11, fontweight='bold', labelpad=8, color='black')

ax.set_xlim(date_series[0], date_series[-1])
ax.grid(True, linestyle=':', alpha=0.5, color='gray')

# Legend hộp trắng viền đen nổi bật
ax.legend(fontsize=10, loc='upper left', frameon=True, facecolor='white', framealpha=0.95, edgecolor='black', shadow=True)

plt.tight_layout()
plt.show()

print("🎉 Hoàn tất vẽ biểu đồ và so sánh đầy đủ cả R1, R2, R3, FinRL PG và Equal Weight!")

In [ ]:
# ====================================================================
# CELL LƯU TẤT CẢ KẾT QUẢ, CHECKPOINTS & NÉN TẢI VỀ (ĐÃ SỬA LỖI STATE_DICT)
# ====================================================================

import os
import shutil
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import FileLink, display, HTML

# 1. Tạo thư mục chứa toàn bộ sản phẩm xuất xưởng
OUTPUT_DIR = "/kaggle/working/portfoliozero_results"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("=" * 80)
print(f"📦 BẮT ĐẦU ĐÓNG GÓI TOÀN BỘ KẾT QUẢ THỰC NGHIỆM VÀO: {OUTPUT_DIR}")
print("=" * 80)

# --------------------------------------------------------------------
# 2. LƯU BẢNG KẾT QUẢ SO SÁNH (CSV)
# --------------------------------------------------------------------
csv_path = os.path.join(OUTPUT_DIR, "backtest_comparison_results.csv")
if 'comparison_df' in globals() and comparison_df is not None:
    comparison_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"  ✓ Đã lưu bảng số liệu so sánh: {csv_path}")
elif 'df_summary' in globals() and df_summary is not None:
    df_summary.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"  ✓ Đã lưu bảng số liệu so sánh: {csv_path}")
else:
    print("  ⚠️ Không tìm thấy biến bảng số liệu để lưu CSV.")

# --------------------------------------------------------------------
# 3. LƯU HÌNH ẢNH BIỂU ĐỒ ĐỘ PHÂN GIẢI CAO (PNG 300 DPI)
# --------------------------------------------------------------------
chart_path = os.path.join(OUTPUT_DIR, "lapv_comparison_chart.png")

fig, ax = plt.subplots(figsize=(14, 6.8), dpi=300)
ax.axhline(0, color='black', linestyle='-', linewidth=1.0, alpha=0.7)

# Đường Equal Weight
ax.plot(date_series, lapv_ew, label='Equal Weight (Baseline)', color='#7f7f7f', linestyle='--', linewidth=1.6)

# Đường FinRL PG
if 'pv_finrl' in globals() and pv_finrl is not None:
    ax.plot(date_series, lapv_finrl, label='FinRL PG (Classic DRL / EIIE)', color='#1f77b4', linestyle='-.', linewidth=1.8)

# Đường R1
if 'pv_r1' in globals() and pv_r1 is not None:
    ax.plot(date_series, lapv_r1, label='PortfolioZero_R1 (Base Log Return)', color='#2ca02c', linestyle=':', linewidth=1.8)

# Đường R2
if 'pv_r2' in globals() and pv_r2 is not None:
    ax.plot(date_series, lapv_r2, label='PortfolioZero_R2 (+ R_alpha Sentiment)', color='#ff7f0e', linestyle='-.', linewidth=2.0)

# Đường R3
if 'pv_r3' in globals() and pv_r3 is not None:
    ax.plot(date_series, lapv_r3, label='PortfolioZero_R3 (+ R_alpha + R_beta)', color='#d62728', linestyle='-', linewidth=2.5)

# Định dạng viền trục và ticks
for spine in ['left', 'bottom', 'top', 'right']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_color('black')
    ax.spines[spine].set_linewidth(1.2)

ax.tick_params(axis='both', which='major', direction='inout', length=6, width=1.2, colors='black', labelsize=10)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate(rotation=30, ha='right')

ax.set_title("Logarithmic Accumulation of Portfolio Value (LAPV) - Out-of-Sample 2023\n10 US Equities (GILD, NVDA, TSLA, MRK, KO, AAPL, GS, WMT, FDX, AMT)", 
             fontsize=13, fontweight='bold', pad=14, color='black')
ax.set_xlabel("Date (Year-Month-Day)", fontsize=11, fontweight='bold', labelpad=8, color='black')
ax.set_ylabel("LAPV: ln(P_t / P_0)", fontsize=11, fontweight='bold', labelpad=8, color='black')
ax.set_xlim(date_series[0], date_series[-1])
ax.grid(True, linestyle=':', alpha=0.5, color='gray')
ax.legend(fontsize=10, loc='upper left', frameon=True, facecolor='white', framealpha=0.95, edgecolor='black', shadow=True)

plt.tight_layout()
fig.savefig(chart_path, dpi=300)
plt.close(fig)
print(f"  ✓ Đã lưu biểu đồ sắc nét (300 DPI): {chart_path}")

# --------------------------------------------------------------------
# 4. LƯU TẤT CẢ CHECKPOINTS MODEL (R1, R2, R3, FinRL PG)
# --------------------------------------------------------------------
def save_single_checkpoint(name, h_net, f_net, g_net):
    ckpt_file = os.path.join(CHECKPOINT_DIR, f"{name}.pth")
    torch.save({
        'model_name': name,
        'h_net_state_dict': h_net.state_dict(),
        'f_net_state_dict': f_net.state_dict(),
        'g_net_state_dict': g_net.state_dict(),
        'target_tickers': TARGET_TICKERS,
        'time_window': TIME_WINDOW,
        'features': FEATURES,
        'hidden_dim': H_DIM
    }, ckpt_file)
    print(f"  ✓ Đã lưu checkpoint {name}: {ckpt_file}")

# Lưu R1
if 'h_r1' in globals() and h_r1 is not None:
    save_single_checkpoint("portfoliozero_r1", h_r1, f_r1, g_r1)

# Lưu R2
if 'h_r2' in globals() and h_r2 is not None:
    save_single_checkpoint("portfoliozero_r2", h_r2, f_r2, g_r2)

# Lưu R3
if 'h_r3' in globals() and h_r3 is not None:
    save_single_checkpoint("portfoliozero_r3", h_r3, f_r3, g_r3)

# Lưu FinRL PG (SỬA LỖI: Lấy state_dict từ mạng train_policy bên trong)
if 'trained_finrl_model' in globals() and trained_finrl_model is not None:
    pg_file = os.path.join(CHECKPOINT_DIR, "finrl_pg_model.pth")
    if hasattr(trained_finrl_model, 'train_policy'):
        torch.save(trained_finrl_model.train_policy.state_dict(), pg_file)
    elif hasattr(trained_finrl_model, 'policy'):
        torch.save(trained_finrl_model.policy.state_dict(), pg_file)
    else:
        torch.save(trained_finrl_model, pg_file)
    print(f"  ✓ Đã lưu checkpoint FinRL PG: {pg_file}")

# --------------------------------------------------------------------
# 5. NÉN TOÀN BỘ VÀO 1 FILE ZIP DUY NHẤT
# --------------------------------------------------------------------
zip_archive_name = "/kaggle/working/portfoliozero_experiments_bundle"
shutil.make_archive(zip_archive_name, 'zip', OUTPUT_DIR)
final_zip_path = zip_archive_name + ".zip"
zip_size_mb = os.path.getsize(final_zip_path) / (1024 * 1024)

print("\n" + "=" * 80)
print(f"🎉 ĐÃ ĐÓNG GÓI THÀNH CÔNG FILE ZIP: {final_zip_path}")
print(f" - Dung lượng file nén: {zip_size_mb:.2f} MB")
print("=" * 80)

# --------------------------------------------------------------------
# 6. HIỂN THỊ LINK & NÚT BẤM TẢI VỀ TRỰC TIẾP
# --------------------------------------------------------------------
display(HTML(f"""
    <div style="background-color: #e6f7ff; border: 2px solid #1890ff; border-radius: 8px; padding: 15px; margin: 15px 0; text-align: center;">
        <h3 style="color: #0050b3; margin-top: 0;">📥 Tải về trọn bộ kết quả & Checkpoints</h3>
        <p style="color: #333; margin-bottom: 15px;">Gói nén gồm: Biểu đồ chất lượng cao (300 DPI), file CSV các chỉ số tài chính và các checkpoint mô hình (.pth).</p>
        <a href="portfoliozero_experiments_bundle.zip" download="portfoliozero_experiments_bundle.zip" 
           style="background-color: #1890ff; color: white; padding: 10px 20px; font-size: 16px; font-weight: bold; text-decoration: none; border-radius: 5px; display: inline-block;">
           ⬇️ BẤM VÀO ĐÂY ĐỂ TẢI XUỐNG (.ZIP)
        </a>
    </div>
"""))

FileLink("portfoliozero_experiments_bundle.zip")